# Prospective Memory DDPM Analysis

This notebook implements a standalone, reproducible pipeline for analyzing prospective memory representations in a DDPM model. Each section is clearly separated and documented for legibility and ease of use.

**Outline:**
1. Setup: environment, imports, constants
2. Sample data: generate or load minimal dataset
3. Core functions: main logic and helpers
4. Main execution cell: script-style entry point
5. Argument parsing: CLI with argparse
6. Unit tests: pytest/unittest
7. Logging & error handling
8. Quick demos / example runs
9. Visualization: plot outputs
10. Save & export: write outputs
11. Performance: profiling and optimization


## 1. Setup: Environment, Imports, Constants

Install and import required libraries. Define constants and configuration variables for the analysis.

In [1]:
# Install required packages if missing (uncomment if needed)
# !pip install numpy matplotlib scikit-learn pandas pytest

import os
import sys
import logging
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from pathlib import Path
import colorsys
import pickle
import json
import torch
from purias_utils.multiitem_working_memory.util.circle_utils import polar2cart

# Get absolute paths
REPO_ROOT = Path('/scratch3/shaiq_home/repos/behaviour_ddpm')
DATA_DIR = REPO_ROOT / 'ddpm' / 'analysis' / 'new_analysis'

# Ensure repo root is importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Model and analysis parameters
PREP_IDX = 2  # Start of delay 1
NEURAL_DIM = 16  # Dimensionality of extracted neural state
N_TRIALS = 200  # Number of non-swap trials to generate
ANGLE_STEP = 30  # Angle step in degrees
SEED = 42
np.random.seed(SEED)

# Print to verify paths
print(f"Working directory: {os.getcwd()}")
print(f"Data directory: {DATA_DIR}")
print(f"sys.path includes repo root: {str(REPO_ROOT) in sys.path}")


Working directory: /scratch3/shaiq_home/repos/behaviour_ddpm/ddpm/analysis/new_analysis
Data directory: /scratch3/shaiq_home/repos/behaviour_ddpm/ddpm/analysis/new_analysis
sys.path includes repo root: True


In [2]:
NULLSPACE_JSON_PATH = REPO_ROOT / "results_link_sampler/index_cued_first_diffusion_0.3_swap_7/nullspace_and_projection.json"
TEACHER_ARGS_PATH = REPO_ROOT / "results_link_sampler/index_cued_first_diffusion_0.3_swap_7/args.yaml"
TEACHER_CKPT_PATH = REPO_ROOT / "results_link_sampler/index_cued_first_diffusion_0.3_swap_7/state.mdl"

# Which nullspace direction (0–13) to ablate for the single-run default
ABLATION_DIRECTION_IDX = 0
# Which student to analyse for the single-run default (0–13)
STUDENT_ID = 8

MODEL_RUN_CONFIGS = {
    f"ablated_teacher_dir_{ABLATION_DIRECTION_IDX:02d}": {
        "args_path": TEACHER_ARGS_PATH,
        "checkpoint_path": TEACHER_CKPT_PATH,
        "nullspace_json_path": NULLSPACE_JSON_PATH,
        "ablation_direction_idx": ABLATION_DIRECTION_IDX,
    },
    f"student_recovery_{STUDENT_ID}": {
        "args_path": REPO_ROOT / f"results_link_sampler/index_cued_first_diffusion_0.3_swap_recovery_{STUDENT_ID}/args.yaml",
        "checkpoint_path": REPO_ROOT / f"results_link_sampler/index_cued_first_diffusion_0.3_swap_recovery_{STUDENT_ID}/state.mdl",
        "nullspace_json_path": None,
        "ablation_direction_idx": None,
    },
}


In [3]:
import sys
from pathlib import Path

_REPO_ROOT = Path('/scratch3/shaiq_home/repos/behaviour_ddpm')
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from ddpm.utils.loading import generate_model_and_task_from_args_path_multiepoch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def _load_single_nullspace_vector(json_path, direction_idx, device):
    """Load one nullspace vector [D] by index from the JSON file."""
    with open(json_path) as f:
        data = json.load(f)
    vectors = data["nullspace_vectors"]["vectors"]
    sorted_keys = sorted(vectors.keys())
    key = sorted_keys[direction_idx]
    vec = np.array(vectors[key], dtype=np.float32)
    print(f"  ablation:   direction {direction_idx} ('{key}'), shape {vec.shape}")
    return torch.tensor(vec, device=device)


LOADED_MODELS = {}
EXTRACTED_RUN_DATA = {}
ANALYSIS_RUN_ORDER = list(MODEL_RUN_CONFIGS.keys())
DEFAULT_ACTIVE_RUN = ANALYSIS_RUN_ORDER[0]

task = model = ACTIVE_ABLATION_VECTOR = RESULTS_DIR = RUN_LABEL = None
neural_states = metadata = None


def activate_run(run_name, restore_extracted=True):
    global task, model, ACTIVE_ABLATION_VECTOR, RESULTS_DIR, RUN_LABEL, neural_states, metadata
    cfg = LOADED_MODELS[run_name]
    task = cfg["task"]
    model = cfg["model"]
    ACTIVE_ABLATION_VECTOR = cfg["ablation_matrix"]
    RESULTS_DIR = cfg["results_dir"]
    RUN_LABEL = run_name
    if restore_extracted and run_name in EXTRACTED_RUN_DATA:
        neural_states = EXTRACTED_RUN_DATA[run_name]["neural_states"]
        metadata = EXTRACTED_RUN_DATA[run_name]["metadata"]
    print(f'Active run: {run_name}')
    print(f'  Results directory: {RESULTS_DIR}')
    abl = ACTIVE_ABLATION_VECTOR
    if abl is not None:
        print(f'  Ablation: single direction, shape {tuple(abl.shape)}')
    else:
        print(f'  Ablation: none')


def extract_neural_state_from_model(trial, task, model, device, prep_idx=2, ablation_vector=None, neural_dim=16):
    with torch.no_grad():
        probe_features = torch.tensor([[trial['color1_angle'], trial['color2_angle']]]) * (np.pi / 180)
        report_features = torch.tensor([[trial['color1_angle'], trial['color2_angle']]]) * (np.pi / 180)
        override_stimulus_features = {
            'probe_features': probe_features,
            'report_features': report_features,
        }
        override_stimulus_cart_features = {
            f'{k}_cart': torch.stack(polar2cart(1.0, v), -1)
            for k, v in override_stimulus_features.items()
        }
        override_stimulus_features_dict = {
            **override_stimulus_features,
            **override_stimulus_cart_features,
            'cued_item_idx': torch.tensor([trial['cue'] - 1]),
        }
        task_variable_dict = task.task_variable_gen.generate_variable_dict(
            batch_size=1, override_stimulus_features_dict=override_stimulus_features_dict
        )
        trial_info = task.generate_trial_information(
            batch_size=1, num_samples=1, override_task_variable_information=task_variable_dict
        )
        prep_inputs = [inp.to(device) if isinstance(inp, torch.Tensor) else inp for inp in trial_info.prep_network_inputs]
        diff_inputs = [inp.to(device) if isinstance(inp, torch.Tensor) else inp for inp in trial_info.diffusion_network_inputs]
        sample_kwargs = {
            'prep_network_inputs': prep_inputs,
            'diffusion_network_inputs': diff_inputs,
            'prep_epoch_durations': trial_info.prep_epoch_durations,
            'diffusion_epoch_durations': trial_info.diffusion_epoch_durations,
            'samples_shape': [1, 1],
            'noise_scaler': 1.0,
        }
        if ablation_vector is not None:
            sample_kwargs['ablation_vector'] = ablation_vector
        prep_dicts, _ = model.generate_samples(**sample_kwargs)
        return prep_dicts[prep_idx]['postprep_state'][0, 0, :neural_dim].cpu().numpy()


print('Infrastructure ready. Run the next cell to load MODEL_RUN_CONFIGS, or skip it and go straight to the batch cell.')


Using device: cuda
Loading run 'ablated_teacher_dir_00'
  args:       /scratch3/shaiq_home/repos/behaviour_ddpm/results_link_sampler/index_cued_first_diffusion_0.3_swap_7/args.yaml
  checkpoint: /scratch3/shaiq_home/repos/behaviour_ddpm/results_link_sampler/index_cued_first_diffusion_0.3_swap_7/state.mdl
  ablation:   direction 0 ('nullspace_00'), shape (16,)
Loading run 'student_recovery_8'
  args:       /scratch3/shaiq_home/repos/behaviour_ddpm/results_link_sampler/index_cued_first_diffusion_0.3_swap_recovery_8/args.yaml
  checkpoint: /scratch3/shaiq_home/repos/behaviour_ddpm/results_link_sampler/index_cued_first_diffusion_0.3_swap_recovery_8/state.mdl
  ablation:   none
Active run: ablated_teacher_dir_00
  Results directory: /scratch3/shaiq_home/repos/behaviour_ddpm/ddpm/analysis/new_analysis/results/prospective_memory_dual/ablated_teacher_dir_00
  Ablation: single direction, shape (16,)

Loaded dual-run contexts: ['ablated_teacher_dir_00', 'student_recovery_8']
Use activate_run('ab

In [ ]:
# Load the two runs defined in MODEL_RUN_CONFIGS (teacher + one student).
# Skip this cell if you are going straight to the auto-discover batch cell.
for run_name, cfg in MODEL_RUN_CONFIGS.items():
    if not cfg['args_path'].exists() or not cfg['checkpoint_path'].exists():
        print(f"Skipping '{run_name}': files not found")
        continue
    print('=' * 80)
    print(f"Loading run '{run_name}'")
    print(f"  args:       {cfg['args_path']}")
    print(f"  checkpoint: {cfg['checkpoint_path']}")
    _, task_r, model_r, _, _ = generate_model_and_task_from_args_path_multiepoch(
        str(cfg['args_path']), device
    )
    ckpt = torch.load(cfg['checkpoint_path'], map_location=device, weights_only=True)
    model_r.load_state_dict(ckpt)
    model_r.eval()
    ablation_vec = None
    if cfg['nullspace_json_path'] is not None:
        ablation_vec = _load_single_nullspace_vector(
            cfg['nullspace_json_path'], cfg['ablation_direction_idx'], device
        )
    else:
        print(f'  ablation:   none')
    results_dir = DATA_DIR / 'results' / 'prospective_memory_dual' / run_name
    results_dir.mkdir(parents=True, exist_ok=True)
    LOADED_MODELS[run_name] = {
        'model': model_r,
        'task': task_r,
        'ablation_matrix': ablation_vec,
        'results_dir': results_dir,
    }

if LOADED_MODELS:
    activate_run(next(iter(LOADED_MODELS)), restore_extracted=False)
    print(f"\nLoaded: {list(LOADED_MODELS.keys())}")
else:
    print('No runs loaded (all files missing). Proceed to the batch cell.')


In [4]:
# Cell: Section 3 - Trial Generation
"""
## 3. Generate Trial Combinations (Non-Swap Trials)

Generate all combinations of cue, color1, and color2 for analysis.
"""

def generate_trial_combinations(angle_step=30):
    """
    Generate all non-swap trial combinations.
    
    Args:
        angle_step: Step size for color angles in degrees
        
    Returns:
        List of trial dictionaries with cue, color1_angle, color2_angle
    """
    angles = list(range(0, 360, angle_step))
    trials = []
    
    for cue in [1, 2]:
        for color1 in angles:
            for color2 in angles:
                trials.append({
                    'cue': cue,
                    'color1_angle': color1,
                    'color2_angle': color2,
                    'swap': False
                })
    return trials

trials = generate_trial_combinations()

In [5]:
def extract_states_for_run(run_name, prep_idx=2, force=False):
    """Extract and store neural states for one configured run (always fresh extraction)."""
    activate_run(run_name, restore_extracted=False)
    print(f"Extracting neural states for run='{run_name}', prep_idx={prep_idx}...")
    print(f"Total trials to process: {len(trials)}")

    neural_states_run = []
    metadata_run = []

    for i, trial in enumerate(trials):
        if i % 50 == 0:
            print(f"  [{run_name}] trial {i}/{len(trials)}")
        state = extract_neural_state_from_model(
            trial, task, model, device,
            prep_idx=prep_idx,
            ablation_vector=ACTIVE_ABLATION_VECTOR,
        )
        neural_states_run.append(state)
        metadata_run.append([trial['cue'], trial['color1_angle'], trial['color2_angle']])

    neural_states_run = np.asarray(neural_states_run, dtype=np.float32)
    metadata_run = np.asarray(metadata_run, dtype=np.float32)

    EXTRACTED_RUN_DATA[run_name] = {
        'prep_idx': prep_idx,
        'neural_states': neural_states_run,
        'metadata': metadata_run,
    }

    activate_run(run_name, restore_extracted=True)
    print(f"\u2713 [{run_name}] extracted {len(neural_states_run)} states with shape {neural_states_run.shape}")
    return neural_states_run, metadata_run


Active run: ablated_teacher_dir_00
  Results directory: /scratch3/shaiq_home/repos/behaviour_ddpm/ddpm/analysis/new_analysis/results/prospective_memory_dual/ablated_teacher_dir_00
  Ablation: single direction, shape (16,)
Extracting neural states for run='ablated_teacher_dir_00', prep_idx=2...
Total trials to process: 288
  [ablated_teacher_dir_00] trial 0/288
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEVER BE DONE IN TRAINING
USING PRECUED ITEMS - SHOULD NEV

In [ ]:
# Runs extraction for MODEL_RUN_CONFIGS runs only.
# SKIP this cell if using the auto-discover batch cell.
for _run_name in ANALYSIS_RUN_ORDER:
    if _run_name not in LOADED_MODELS:
        print(f"Skipping '{_run_name}': not in LOADED_MODELS")
        continue
    extract_states_for_run(_run_name, prep_idx=PREP_IDX, force=True)

if LOADED_MODELS:
    activate_run(DEFAULT_ACTIVE_RUN, restore_extracted=True)
    print(f"Current active run: {RUN_LABEL}")


In [ ]:
def use_teacher():
    run_name = f"ablated_teacher_dir_{ABLATION_DIRECTION_IDX:02d}"
    activate_run(run_name, restore_extracted=True)


def use_student():
    run_name = f"student_recovery_{STUDENT_ID}"
    activate_run(run_name, restore_extracted=True)


print("Workflow for two full plot sets:")
print("  1) Run use_teacher(), then execute analysis/plot cells below.")
print("  2) Run use_student(), then execute the same analysis/plot cells again.")
print("Each run writes to its own RESULTS_DIR, so filenames do not collide.")


In [6]:

# ── Full analysis function (self-contained, runs for whatever is active) ─────

def run_full_analysis_for_active_run():
    """Run all analysis + save all plots for the currently active run.

    Reads globals: task, model, device, ACTIVE_ABLATION_VECTOR, RESULTS_DIR,
                   RUN_LABEL, neural_states, metadata, trials.
    """
    import colorsys as _col
    from matplotlib.lines import Line2D as _L2D
    from sklearn.decomposition import PCA as _PCA

    _NB = globals().get('N_BINS', 12)

    print(f"\n{'═'*72}\nFULL ANALYSIS: {RUN_LABEL}\n{'═'*72}")

    # ── local helpers ─────────────────────────────────────────────────────────

    def _hue(b, n=12):
        return _col.hsv_to_rgb(b / n, 0.9, 0.9)

    def _bangle(a, sz=30.0):
        return int(a // sz) % int(360 // sz)

    def _fit(data):
        ctr = data.mean(0)
        p = _PCA(n_components=3).fit(data - ctr)
        n = p.components_[2]; n /= np.linalg.norm(n)
        return n, ctr, float(p.explained_variance_ratio_[:2].sum()), p.explained_variance_ratio_

    def _angle_deg(n1, n2):
        return float(np.degrees(np.arccos(np.clip(abs(np.dot(n1, n2)), 0, 1))))

    def _bin_cued(states, meta, nb=12):
        sz = 360.0 / nb
        bd = {c: {b: [] for b in range(nb)} for c in [1, 2]}
        for i, (cue, c1, c2) in enumerate(meta):
            cue = int(cue)
            bd[cue][_bangle(c1 if cue == 1 else c2, sz)].append(states[i])
        av = {c: np.full((nb, states.shape[1]), np.nan) for c in [1, 2]}
        for c in [1, 2]:
            for b, sl in bd[c].items():
                if sl: av[c][b] = np.mean(sl, 0)
        return av

    def _avg_role(states_t, meta_a, cue, role, nb=12):
        sz = 360.0 / nb
        mask = meta_a[:, 0].astype(int) == cue
        st, mt = states_t[mask], meta_a[mask]
        bins = {b: [] for b in range(nb)}
        for s, (_, c1, c2) in zip(st, mt):
            a = (c1 if cue == 1 else c2) if role == 'target' else (c2 if cue == 1 else c1)
            bins[_bangle(a, sz)].append(s)
        out = [np.mean(bins[b], 0) for b in range(nb) if bins[b]]
        return np.array(out, np.float32) if out else np.empty((0, states_t.shape[1]), np.float32)

    def _pc12v(x):
        if x.shape[0] < 3: return np.nan
        p = _PCA(n_components=min(3, x.shape[0], x.shape[1])).fit(x)
        return float(p.explained_variance_ratio_[:2].sum())

    def _ts_metrics(st, ma, cue, nb=12):
        ta = _avg_role(st, ma, cue, 'target', nb)
        da = _avg_role(st, ma, cue, 'distractor', nb)
        nan = dict(plane_angle_deg=np.nan, planarity_cos=np.nan,
                   centroid_separation=np.nan, target_var_pc12=np.nan,
                   distractor_var_pc12=np.nan, combined_var_pc12=np.nan,
                   target_planarity=np.nan, distractor_planarity=np.nan,
                   target_centroid_norm=np.nan, distractor_centroid_norm=np.nan,
                   target_mean_radius=np.nan, target_radius_std=np.nan,
                   distractor_mean_radius=np.nan, distractor_radius_std=np.nan,
                   target_eccentricity=np.nan, distractor_eccentricity=np.nan,
                   target_arc_std=np.nan, distractor_arc_std=np.nan,
                   ring_separation_dprime=np.nan,
                   overall_mean_radius=np.nan, overall_radius_std=np.nan,
                   overall_eccentricity=np.nan, overall_arc_std=np.nan,
                   ring_planarity=np.nan)
        if ta.shape[0] < 3 or da.shape[0] < 3: return nan
        comb = np.vstack([ta, da])
        pc = _PCA(n_components=3); coords = pc.fit_transform(comb)
        tn_v, tc, tpl, _ = _fit(coords[:ta.shape[0]])
        dn_v, dc, dpl, _ = _fit(coords[ta.shape[0]:])
        theta = _angle_deg(tn_v, dn_v)

        t_pts = coords[:ta.shape[0]]
        d_pts = coords[ta.shape[0]:]

        t_dists = np.linalg.norm(t_pts - tc, axis=1)
        d_dists = np.linalg.norm(d_pts - dc, axis=1)
        t_mean_r = float(t_dists.mean())
        d_mean_r = float(d_dists.mean())
        t_rad_std = float(t_dists.std())
        d_rad_std = float(d_dists.std())

        def _ecc(pts):
            ev = _PCA(n_components=2).fit(pts).explained_variance_
            return float(ev[0] / ev[1]) if ev[1] > 0 else np.nan
        t_ecc = _ecc(t_pts)
        d_ecc = _ecc(d_pts)

        def _arc_std(pts):
            arcs = [np.linalg.norm(pts[(i + 1) % len(pts)] - pts[i]) for i in range(len(pts))]
            return float(np.std(arcs))
        t_arc_std = _arc_std(t_pts)
        d_arc_std = _arc_std(d_pts)

        denom = np.sqrt(0.5 * (t_mean_r ** 2 + d_mean_r ** 2))
        dprime = float(np.linalg.norm(tc - dc) / denom) if denom > 0 else np.nan

        # Overall aggregated metrics (cue-level, not separated by target/distractor)
        overall_mean_r = float(0.5 * (t_mean_r + d_mean_r))
        overall_rad_std = float(0.5 * (t_rad_std + d_rad_std))
        overall_ecc = float(0.5 * (t_ecc + d_ecc)) if not (np.isnan(t_ecc) or np.isnan(d_ecc)) else np.nan
        overall_arc = float(0.5 * (t_arc_std + d_arc_std))
        ring_plan = float(0.5 * (tpl + dpl))  # Planarity of the rings (how flat)

        return dict(plane_angle_deg=theta,
                    planarity_cos=float(abs(np.cos(np.radians(theta)))),
                    centroid_separation=float(np.linalg.norm(tc - dc)),
                    target_var_pc12=_pc12v(ta), distractor_var_pc12=_pc12v(da),
                    combined_var_pc12=float(pc.explained_variance_ratio_[:2].sum()),
                    target_planarity=tpl, distractor_planarity=dpl,
                    target_centroid_norm=float(np.linalg.norm(tc)),
                    distractor_centroid_norm=float(np.linalg.norm(dc)),
                    target_mean_radius=t_mean_r, target_radius_std=t_rad_std,
                    distractor_mean_radius=d_mean_r, distractor_radius_std=d_rad_std,
                    target_eccentricity=t_ecc, distractor_eccentricity=d_ecc,
                    target_arc_std=t_arc_std, distractor_arc_std=d_arc_std,
                    ring_separation_dprime=dprime,
                    overall_mean_radius=overall_mean_r, overall_radius_std=overall_rad_std,
                    overall_eccentricity=overall_ecc, overall_arc_std=overall_arc,
                    ring_planarity=ring_plan)

    def _get_timeline(trial):
        with torch.no_grad():
            pf = torch.tensor([[trial['color1_angle'], trial['color2_angle']]]) * (np.pi / 180)
            osf = {'probe_features': pf, 'report_features': pf.clone()}
            osfc = {f'{k}_cart': torch.stack(polar2cart(1.0, v), -1) for k, v in osf.items()}
            osfd = {**osf, **osfc, 'cued_item_idx': torch.tensor([trial['cue'] - 1])}
            tvd = task.task_variable_gen.generate_variable_dict(1, override_stimulus_features_dict=osfd)
            ti = task.generate_trial_information(1, 1, override_task_variable_information=tvd)
            pni = [x.to(device) if isinstance(x, torch.Tensor) else x for x in ti.prep_network_inputs]
            dni = [x.to(device) if isinstance(x, torch.Tensor) else x for x in ti.diffusion_network_inputs]
            kw = dict(prep_network_inputs=pni, diffusion_network_inputs=dni,
                      prep_epoch_durations=ti.prep_epoch_durations,
                      diffusion_epoch_durations=ti.diffusion_epoch_durations,
                      samples_shape=[1, 1], noise_scaler=1.0)
            if ACTIVE_ABLATION_VECTOR is not None:
                kw['ablation_vector'] = ACTIVE_ABLATION_VECTOR
            pd_, sd = model.generate_samples(**kw)
            segs = [pd_[i]['preparatory_trajectory'][0, 0, :, :16].cpu().numpy().astype(np.float32)
                    for i in range(len(pd_))]
            pep = [int(s.shape[0]) for s in segs]
            key = 'embedded_sample_trajectory' if 'embedded_sample_trajectory' in sd else 'sample_trajectory'
            diff = sd[key][0, 0, :, :16].cpu().numpy().astype(np.float32)
            return np.concatenate(segs + [diff], 0), pep, int(diff.shape[0])

    # ── 1. PCA viz: target vs distractor ─────────────────────────────────────
    print("1/4  PCA visualization (target vs distractor) …")
    sz = 360.0 / _NB
    btd = {c: {} for c in [1, 2]}
    for i, (cue, c1, c2) in enumerate(metadata):
        cue = int(cue)
        ta, da = (c1, c2) if cue == 1 else (c2, c1)
        key = (_bangle(ta, sz), _bangle(da, sz))
        btd[cue].setdefault(key, []).append(neural_states[i])
    atd = {c: {k: np.mean(v, 0) for k, v in btd[c].items()} for c in [1, 2]}

    tgt = {b: np.mean([s for c in [1,2] for (ft,fd),s in atd[c].items() if ft==b], 0)
           for b in range(_NB)}
    dst = {b: np.mean([s for c in [1,2] for (ft,fd),s in atd[c].items() if fd==b], 0)
           for b in range(_NB)}
    cst = np.array([tgt[b] for b in sorted(tgt)] + [dst[b] for b in sorted(dst)])
    clb = np.array([(0,b) for b in sorted(tgt)] + [(1,b) for b in sorted(dst)])
    pcac = _PCA(n_components=3); cpca = pcac.fit_transform(cst)
    tm, dm = clb[:,0]==0, clb[:,0]==1
    tp, tb_ = cpca[tm], clb[tm,1].astype(int)
    dp, db_ = cpca[dm], clb[dm,1].astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{RUN_LABEL} — Target vs Distractor", fontsize=13, fontweight='bold')
    ax3d = fig.add_subplot(131, projection='3d')
    for i in range(len(tp)):
        ax3d.scatter(*tp[i], c=[_hue(tb_[i])], marker='o', s=150, alpha=0.8, edgecolors='k', linewidth=2)
    for i in range(len(dp)):
        ax3d.scatter(*dp[i], c=[_hue(db_[i])], marker='^', s=120, alpha=0.8, edgecolors='grey', linewidth=1.5)
    ax3d.set_xlabel(f'PC1 ({pcac.explained_variance_ratio_[0]:.1%})', fontsize=11)
    ax3d.set_ylabel(f'PC2 ({pcac.explained_variance_ratio_[1]:.1%})', fontsize=11)
    ax3d.set_zlabel(f'PC3 ({pcac.explained_variance_ratio_[2]:.1%})', fontsize=11)
    ax3d.set_title('3D: ○ Target, △ Distractor', fontsize=11)
    for ax_2d, (xi, yi), xlabel, ylabel, title in [
        (plt.subplot(132), (0,1), f'PC1 ({pcac.explained_variance_ratio_[0]:.1%})', f'PC2 ({pcac.explained_variance_ratio_[1]:.1%})', 'PC1 vs PC2'),
        (plt.subplot(133), (1,2), f'PC2 ({pcac.explained_variance_ratio_[1]:.1%})', f'PC3 ({pcac.explained_variance_ratio_[2]:.1%})', 'PC2 vs PC3'),
    ]:
        for i in range(len(tp)):
            ax_2d.scatter(tp[i,xi], tp[i,yi], c=[_hue(tb_[i])], marker='o', s=150, alpha=0.8, edgecolors='k', linewidth=2)
        for i in range(len(dp)):
            ax_2d.scatter(dp[i,xi], dp[i,yi], c=[_hue(db_[i])], marker='^', s=120, alpha=0.8, edgecolors='grey', linewidth=1.5)
        ax_2d.set_xlabel(xlabel, fontsize=11); ax_2d.set_ylabel(ylabel, fontsize=11)
        ax_2d.set_title(title, fontsize=11); ax_2d.grid(True, alpha=0.3); ax_2d.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / '2d_binning_target_vs_distractor_combined.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── 2. Prep epoch analysis + visualization ────────────────────────────────
    print("2/4  Prep epoch analysis …")
    _pt_names = {0:"prep_idx=0: End of CUE\n(Start of Delay 1)",
                 1:"prep_idx=1: Before Stimulus\n(End of Delay 1)",
                 2:"prep_idx=2: End of STIMULUS\n(Start of Delay 2)",
                 3:"prep_idx=3: Before Response"}
    apr = {}
    for pidx in [0, 1, 2, 3]:
        sp, mp = [], []
        for tr in trials:
            sp.append(extract_neural_state_from_model(
                tr, task, model, device, prep_idx=pidx, ablation_vector=ACTIVE_ABLATION_VECTOR))
            mp.append([tr['cue'], tr['color1_angle'], tr['color2_angle']])
        sp, mp = np.array(sp), np.array(mp)
        av = _bin_cued(sp, mp, _NB)
        allp = np.vstack([av[1], av[2]])
        allp = allp[~np.isnan(allp).any(1)]
        pca_p = _PCA(n_components=3); coords = pca_p.fit_transform(allp)
        c1p, c2p = coords[:_NB], coords[_NB:_NB*2]
        n1, ct1, pl1, _ = _fit(c1p); n2, ct2, pl2, _ = _fit(c2p)
        ang = _angle_deg(n1, n2); sep = float(np.linalg.norm(ct1 - ct2))
        apr[pidx] = dict(cue1_pca=c1p, cue2_pca=c2p, cue1_planarity=pl1, cue2_planarity=pl2,
                         plane_angle=ang, separation=sep,
                         explained_variance=pca_p.explained_variance_ratio_)
        print(f"  prep_idx={pidx}: angle={ang:.1f}°, sep={sep:.4f}, planarity=[{pl1:.2%},{pl2:.2%}]")

    cm_handles = [_L2D([0],[0],marker='o',linestyle='None',color='w',markerfacecolor='lightgray',markeredgecolor='k',markersize=7,label='Cue 1'),
                  _L2D([0],[0],marker='^',linestyle='None',color='w',markerfacecolor='lightgray',markeredgecolor='k',markersize=7,label='Cue 2')]
    fig2 = plt.figure(figsize=(18, 20))
    fig2.suptitle(f"{RUN_LABEL} — Prospective Memory Across Prep Epochs", fontsize=14, fontweight='bold')
    for ri, pidx in enumerate([0,1,2,3]):
        res = apr[pidx]; var = res['explained_variance']
        c1c = (np.tile([[0.55,0.55,0.55]],(res['cue1_pca'].shape[0],1)) if pidx < 2
               else np.array([_hue(b % _NB) for b in range(res['cue1_pca'].shape[0])]))
        c2c = (np.tile([[0.55,0.55,0.55]],(res['cue2_pca'].shape[0],1)) if pidx < 2
               else np.array([_hue(b % _NB) for b in range(res['cue2_pca'].shape[0])]))
        a3 = fig2.add_subplot(4,3,ri*3+1,projection='3d')
        a3.scatter(*res['cue1_pca'].T,c=c1c,marker='o',s=80,edgecolors='k',linewidths=1,alpha=0.9)
        a3.scatter(*res['cue2_pca'].T,c=c2c,marker='^',s=80,edgecolors='k',linewidths=1,alpha=0.9)
        a3.set_xlabel(f'PC1 ({var[0]:.1%})',fontsize=9); a3.set_ylabel(f'PC2 ({var[1]:.1%})',fontsize=9)
        a3.set_zlabel(f'PC3 ({var[2]:.1%})',fontsize=9)
        a3.set_title(f"{_pt_names[pidx]}\nSep:{res['separation']:.3f}",fontsize=10,fontweight='bold')
        a3.legend(handles=cm_handles,fontsize=8)
        for col, (xi,yi), tit in [(fig2.add_subplot(4,3,ri*3+2),(0,1),'PC1 vs PC2'),
                                   (fig2.add_subplot(4,3,ri*3+3),(1,2),'PC2 vs PC3')]:
            col.scatter(res['cue1_pca'][:,xi],res['cue1_pca'][:,yi],c=c1c,marker='o',s=80,edgecolors='k',linewidths=1,alpha=0.9)
            col.scatter(res['cue2_pca'][:,xi],res['cue2_pca'][:,yi],c=c2c,marker='^',s=80,edgecolors='k',linewidths=1,alpha=0.9)
            col.set_xlabel(f'PC{xi+1} ({var[xi]:.1%})',fontsize=9,fontweight='bold')
            col.set_ylabel(f'PC{yi+1} ({var[yi]:.1%})',fontsize=9,fontweight='bold')
            col.set_title(tit,fontsize=10,fontweight='bold')
            col.legend(handles=cm_handles,fontsize=8); col.grid(True,alpha=0.3); col.axis('equal')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'prospective_memory_all_prep_indices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {RESULTS_DIR / 'prospective_memory_all_prep_indices.png'}")

    # ── 3. Full timeline ──────────────────────────────────────────────────────
    print("3/4  Full timeline extraction …")
    trows, mrows, pep_ref, ds_ref = [], [], None, None
    for i, tr in enumerate(trials):
        if i % 50 == 0: print(f"  trial {i}/{len(trials)}")
        tl, pep, ds = _get_timeline(tr)
        if pep_ref is None: pep_ref, ds_ref = pep, ds
        trows.append(tl); mrows.append([tr['cue'], tr['color1_angle'], tr['color2_angle']])
    tl_st = np.asarray(trows, np.float32); tl_mt = np.asarray(mrows, np.float32)
    TOTAL = int(tl_st.shape[1])

    # Save raw states so the fidelity cell can bootstrap offline
    _npz_path = RESULTS_DIR / 'timeline_raw_states.npz'
    np.savez_compressed(_npz_path, tl_st=tl_st, tl_mt=tl_mt,
                        pep_ref=np.array(pep_ref, dtype=np.int32),
                        n_diffusion_steps=np.int32(ds_ref))
    print(f'  Saved raw states: {_npz_path}')

    ph_names = [f'prep_idx={i}' for i in range(len(pep_ref))] + ['diffusion']
    ph_bounds = [0]
    cur = 0
    for d in pep_ref: cur += int(d); ph_bounds.append(cur)
    ph_bounds.append(TOTAL)

    print("4/4  Computing timeline metrics …")
    mets = {c: [_ts_metrics(tl_st[:,t,:], tl_mt, c, _NB) for t in range(TOTAL)] for c in [1,2]}

    x = np.arange(TOTAL)
    span_cols = ['#cde5ff','#e8f3ff','#dff7e2','#f9f2d0','#f5e4ff']
    pf_str = ' | '.join([f'prep_idx={i}:{d}' for i,d in enumerate(pep_ref)] + [f'diffusion:{ds_ref}'])

    def _pm(ax):
        for i in range(len(ph_names)):
            ax.axvspan(ph_bounds[i]-.5, ph_bounds[i+1]-.5, color=span_cols[i%len(span_cols)], alpha=0.18, zorder=0)
        for b in ph_bounds[1:-1]: ax.axvline(b-.5, color='k', linestyle='--', linewidth=.9, alpha=.45)
        ctrs = [(ph_bounds[i]+ph_bounds[i+1]-1)/2 for i in range(len(ph_names))]
        sx = ax.secondary_xaxis('top'); sx.set_xticks(ctrs); sx.set_xticklabels(ph_names, fontsize=8)
        ax.set_xlim(-.5, TOTAL-.5)

    def _foot(fig):
        fig.text(.5,.01,f'[{RUN_LABEL}]  {pf_str}',ha='center',fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.25',facecolor='white',alpha=.78,edgecolor='gray'))

    # Figure A: cue1 vs cue2
    c1a=np.array([m['plane_angle_deg'] for m in mets[1]]); c2a=np.array([m['plane_angle_deg'] for m in mets[2]])
    c1p_=np.array([m['planarity_cos'] for m in mets[1]]); c2p_=np.array([m['planarity_cos'] for m in mets[2]])
    c1s=np.array([m['centroid_separation'] for m in mets[1]]); c2s=np.array([m['centroid_separation'] for m in mets[2]])
    c1v=np.array([m['combined_var_pc12'] for m in mets[1]]); c2v=np.array([m['combined_var_pc12'] for m in mets[2]])
    fig3, ax3s = plt.subplots(2,2,figsize=(15,9)); _foot(fig3)
    fig3.suptitle(f"{RUN_LABEL} — Cue 1 vs Cue 2", fontsize=13, fontweight='bold')
    for (r,c),(a1,a2,ttl,yl) in zip([(0,0),(0,1),(1,0),(1,1)],
        [(c1a,c2a,'Plane Angle','Angle (deg)'),(c1p_,c2p_,'Planarity (|cos θ|)','Planarity'),
         (c1s,c2s,'Centroid Separation','L2 sep.'),(c1v,c2v,'PCA Variance (PC1+PC2)','Var. explained')]):
        ax=ax3s[r,c]; ax.plot(x,a1,color='tab:red',lw=2,label='Cue 1'); ax.plot(x,a2,color='tab:blue',lw=2,label='Cue 2')
        ax.set_title(ttl); ax.set_xlabel('Step'); ax.set_ylabel(yl); ax.legend(); ax.grid(alpha=.3); _pm(ax)
    plt.tight_layout(rect=[0,.05,1,1])
    plt.savefig(RESULTS_DIR/'timeline_stats_cue1_vs_cue2.png', dpi=150, bbox_inches='tight'); plt.show()
    print(f"  Saved: {RESULTS_DIR/'timeline_stats_cue1_vs_cue2.png'}")

    # Figure A2: Ring Geometry (Cue 1 vs Cue 2) - aggregated metrics
    c1_omr = np.array([m['overall_mean_radius']     for m in mets[1]])
    c2_omr = np.array([m['overall_mean_radius']     for m in mets[2]])
    c1_ors = np.array([m['overall_radius_std']      for m in mets[1]])
    c2_ors = np.array([m['overall_radius_std']      for m in mets[2]])
    c1_oe  = np.array([m['overall_eccentricity']    for m in mets[1]])
    c2_oe  = np.array([m['overall_eccentricity']    for m in mets[2]])
    c1_oa  = np.array([m['overall_arc_std']         for m in mets[1]])
    c2_oa  = np.array([m['overall_arc_std']         for m in mets[2]])
    c1_rp  = np.array([m['ring_planarity']          for m in mets[1]])
    c2_rp  = np.array([m['ring_planarity']          for m in mets[2]])
    c1_dp  = np.array([m['ring_separation_dprime']  for m in mets[1]])
    c2_dp  = np.array([m['ring_separation_dprime']  for m in mets[2]])

    fig_rg_a, ax_rg_a = plt.subplots(2, 3, figsize=(18, 9)); _foot(fig_rg_a)
    fig_rg_a.suptitle(f"{RUN_LABEL} — Ring Geometry (Cue 1 vs Cue 2)", fontsize=13, fontweight='bold')
    _ring_specs_a = [
        (0, 0, c1_omr,  c2_omr,  'Mean Ring Radius',       'Radius'),
        (0, 1, c1_ors,  c2_ors,  'Ring Radius Std',        'Std (uniformity)'),
        (0, 2, c1_oe,   c2_oe,   'Ring Eccentricity',      'PC1/PC2 ratio'),
        (1, 0, c1_oa,   c2_oa,   'Arc-spacing Std',        'Std of arc lengths'),
        (1, 1, c1_rp,   c2_rp,   'Ring Planarity',         'Variance explained'),
        (1, 2, c1_dp,   c2_dp,   "Ring Separation d'",     "d'"),
    ]
    for r, c, a1, a2, ttl, yl in _ring_specs_a:
        ax = ax_rg_a[r, c]
        ax.plot(x, a1, color='tab:red',  lw=2, label='Cue 1')
        ax.plot(x, a2, color='tab:blue', lw=2, label='Cue 2')
        ax.set_title(ttl); ax.set_xlabel('Step'); ax.set_ylabel(yl)
        ax.legend(); ax.grid(alpha=.3); _pm(ax)
    plt.tight_layout(rect=[0, .05, 1, 1])
    plt.savefig(RESULTS_DIR / 'timeline_ring_geometry_cue1_vs_cue2.png', dpi=150, bbox_inches='tight'); plt.show()
    print(f"  Saved: {RESULTS_DIR / 'timeline_ring_geometry_cue1_vs_cue2.png'}")

    # Figures B2 & C2: Ring Geometry per-cue (Target vs Distractor)
    for cue, ct, cd in [(1,'tab:green','tab:orange'),(2,'tab:purple','tab:brown')]:
        tmr = np.array([m['target_mean_radius']     for m in mets[cue]])
        dmr = np.array([m['distractor_mean_radius'] for m in mets[cue]])
        trs = np.array([m['target_radius_std']      for m in mets[cue]])
        drs = np.array([m['distractor_radius_std']  for m in mets[cue]])
        te  = np.array([m['target_eccentricity']    for m in mets[cue]])
        de  = np.array([m['distractor_eccentricity']for m in mets[cue]])
        ta  = np.array([m['target_arc_std']         for m in mets[cue]])
        da  = np.array([m['distractor_arc_std']     for m in mets[cue]])
        fig_rg_cd, ax_rg_cd = plt.subplots(2, 2, figsize=(15, 9)); _foot(fig_rg_cd)
        fig_rg_cd.suptitle(f"{RUN_LABEL} — Ring Geometry (Cue {cue}: Target vs Distractor)", fontsize=13, fontweight='bold')
        for (r,c),(a1,a2,l1,l2,ttl,yl) in zip([(0,0),(0,1),(1,0),(1,1)],
            [(tmr,dmr,'Target radius','Distractor radius','Mean Ring Radius','Radius'),
             (trs,drs,'Target std','Distractor std','Ring Radius Std','Std (uniformity)'),
             (te,de,'Target eccentricity','Distractor eccentricity','Ring Eccentricity','PC1/PC2 ratio'),
             (ta,da,'Target arc std','Distractor arc std','Arc-spacing Std','Std of arc lengths')]):
            ax=ax_rg_cd[r,c]; ax.plot(x,a1,color=ct,lw=2,label=l1); ax.plot(x,a2,color=cd,lw=2,label=l2)
            ax.set_title(ttl); ax.set_xlabel('Step'); ax.set_ylabel(yl); ax.legend(); ax.grid(alpha=.3); _pm(ax)
        plt.tight_layout(rect=[0,.05,1,1])
        out = RESULTS_DIR/f'timeline_ring_geometry_cue{cue}_target_vs_distractor.png'
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
        print(f"  Saved: {out}")

    # Figures B & C: per-cue target vs distractor
    for cue, ct, cd in [(1,'tab:green','tab:orange'),(2,'tab:purple','tab:brown')]:
        vt=np.array([m['target_var_pc12'] for m in mets[cue]]); vd=np.array([m['distractor_var_pc12'] for m in mets[cue]])
        pt=np.array([m['target_planarity'] for m in mets[cue]]); pd_=np.array([m['distractor_planarity'] for m in mets[cue]])
        nt=np.array([m['target_centroid_norm'] for m in mets[cue]]); nd=np.array([m['distractor_centroid_norm'] for m in mets[cue]])
        ag=np.array([m['plane_angle_deg'] for m in mets[cue]])
        fig4, ax4s = plt.subplots(2,2,figsize=(15,9)); _foot(fig4)
        fig4.suptitle(f"{RUN_LABEL} — Cue {cue}: Target vs Distractor", fontsize=13, fontweight='bold')
        for (r,c),(a1,a2,l1,l2,ttl,yl) in zip([(0,0),(0,1),(1,0),(1,1)],
            [(ag,None,'Plane angle',None,f'Cue {cue}: Plane Angle','Angle (deg)'),
             (pt,pd_,'Target planarity','Distractor planarity',f'Cue {cue}: Planarity','Planarity'),
             (nt,nd,'Target centroid norm','Distractor centroid norm',f'Cue {cue}: Centroid Norm','Norm'),
             (vt,vd,'Target var','Distractor var',f'Cue {cue}: PCA Variance','Var. explained')]):
            ax=ax4s[r,c]; ax.plot(x,a1,color=ct,lw=2,label=l1)
            if a2 is not None: ax.plot(x,a2,color=cd,lw=2,label=l2); ax.legend()
            ax.set_title(ttl); ax.set_xlabel('Step'); ax.set_ylabel(yl); ax.grid(alpha=.3); _pm(ax)
        plt.tight_layout(rect=[0,.05,1,1])
        out = RESULTS_DIR/f'timeline_stats_cue{cue}_target_vs_distractor.png'
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
        print(f"  Saved: {out}")

    print(f"\n✓ [{RUN_LABEL}] Done. Plots in {RESULTS_DIR}")


print("run_full_analysis_for_active_run() defined.")
print("Run the next cell to produce plots for BOTH models automatically.")


run_full_analysis_for_active_run() defined.
Run the next cell to produce plots for BOTH models automatically.


In [ ]:

# ── Run full analysis for BOTH models ────────────────────────────────────────
for _run in ANALYSIS_RUN_ORDER:
    activate_run(_run, restore_extracted=True)
    run_full_analysis_for_active_run()

print("\nAll done. Plots written to:")
for _run in ANALYSIS_RUN_ORDER:
    print(f"  {LOADED_MODELS[_run]['results_dir']}")


In [ ]:
# ── Batch: all 14 teacher ablation directions + all 14 students ──────────────
# Run this cell to generate timeline66 (and all other) plots for every
# ablation direction and every student.  Each run saves to a unique directory:
#   prospective_memory_dual/ablated_teacher_dir_NN/
#   prospective_memory_dual/student_recovery_N/

import gc

_batch_configs = {}
for _dir_idx in range(14):
    _name = f"ablated_teacher_dir_{_dir_idx:02d}"
    _batch_configs[_name] = {
        "args_path": TEACHER_ARGS_PATH,
        "checkpoint_path": TEACHER_CKPT_PATH,
        "nullspace_json_path": NULLSPACE_JSON_PATH,
        "ablation_direction_idx": _dir_idx,
    }
for _sid in range(14):
    _name = f"student_recovery_{_sid}"
    _batch_configs[_name] = {
        "args_path": REPO_ROOT / f"results_link_sampler/index_cued_first_diffusion_0.3_swap_recovery_{_sid}/args.yaml",
        "checkpoint_path": REPO_ROOT / f"results_link_sampler/index_cued_first_diffusion_0.3_swap_recovery_{_sid}/state.mdl",
        "nullspace_json_path": None,
        "ablation_direction_idx": None,
    }

print(f"Batch: {len(_batch_configs)} runs to process")

for _run_name, _cfg in _batch_configs.items():
    print(f"\n{'='*80}\nBATCH: {_run_name}\n{'='*80}")

    # Load model fresh for this run
    _, _task_r, _model_r, _, _ = generate_model_and_task_from_args_path_multiepoch(
        str(_cfg["args_path"]), device
    )
    _ckpt = torch.load(_cfg["checkpoint_path"], map_location=device, weights_only=True)
    _model_r.load_state_dict(_ckpt)
    _model_r.eval()

    _abl_vec = None
    if _cfg["nullspace_json_path"] is not None:
        _abl_vec = _load_single_nullspace_vector(
            _cfg["nullspace_json_path"], _cfg["ablation_direction_idx"], device
        )

    _results_dir = DATA_DIR / "results" / "prospective_memory_dual" / _run_name
    _results_dir.mkdir(parents=True, exist_ok=True)

    # Register in LOADED_MODELS so extract_states_for_run can find it
    LOADED_MODELS[_run_name] = {
        "model": _model_r,
        "task": _task_r,
        "ablation_matrix": _abl_vec,
        "results_dir": _results_dir,
    }

    extract_states_for_run(_run_name, prep_idx=PREP_IDX, force=True)
    run_full_analysis_for_active_run()

    # Release GPU memory before loading the next model
    LOADED_MODELS[_run_name]["model"] = None
    del _model_r, _ckpt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print(f"\u2713 Complete: {_run_name}  \u2192  {_results_dir}")

print("\n\u2713 All batch runs complete.")


In [ ]:
# Extract states at prep_idx=2 and bin by BOTH target and distractor
print("="*60)
print("BINNING BY TARGET AND DISTRACTOR AT prep_idx=2")
print("="*60)

# Restore shared defaults/helpers that may be removed during notebook cleanup.
if "N_BINS" not in globals():
    N_BINS = 12

def bin_angle(angle, bin_size=30.0):
    """Bin an angle into discrete bins."""
    return int(angle // bin_size) % int(360 // bin_size)

def bin_and_average_by_cued_color(states, metadata, n_bins=12):
    """Bin trials by cued color and average neural states per bin and cue."""
    bin_size = 360.0 / n_bins
    binned_data = {
        1: {b: [] for b in range(n_bins)},
        2: {b: [] for b in range(n_bins)}
    }

    for i, (cue, c1, c2) in enumerate(metadata):
        cue = int(cue)
        cued_angle = c1 if cue == 1 else c2
        bin_idx = bin_angle(cued_angle, bin_size)
        binned_data[cue][bin_idx].append(states[i])

    averaged_data = {
        1: np.zeros((n_bins, states.shape[1])),
        2: np.zeros((n_bins, states.shape[1]))
    }

    for cue in [1, 2]:
        for b in range(n_bins):
            if binned_data[cue][b]:
                averaged_data[cue][b] = np.mean(binned_data[cue][b], axis=0)
            else:
                averaged_data[cue][b] = np.nan

    return averaged_data, binned_data

# First, extract states at prep_idx=2 if PREP_IDX is not already 2
if PREP_IDX != 2:
    print(f"\nExtracting additional states at prep_idx=2 (currently using prep_idx={PREP_IDX})...")
    states_prep2 = []
    metadata_prep2 = []
    
    for i, trial in enumerate(trials):
        if i % 100 == 0 and i > 0:
            print(f"  Processing trial {i}/{len(trials)}")
        try:
            state = extract_neural_state_from_model(trial, task, model, device, prep_idx=2)
            states_prep2.append(state)
            metadata_prep2.append([trial['cue'], trial['color1_angle'], trial['color2_angle']])
        except Exception as e:
            print(f"  Trial {i} failed: {e}")
            break
    
    states_prep2 = np.array(states_prep2)
    metadata_prep2 = np.array(metadata_prep2)
    print(f"✓ Extracted {len(states_prep2)} states at prep_idx=2")
else:
    # Use the already extracted states
    print(f"\nUsing already extracted states at prep_idx={PREP_IDX}")
    states_prep2 = neural_states.copy()
    metadata_prep2 = metadata.copy()

print(f"\nTotal states at prep_idx=2: {len(states_prep2)}")
print(f"Shape: {states_prep2.shape}")

# Create bins for both target and distractor
# For cue=1: target=color1, distractor=color2
# For cue=2: target=color2, distractor=color1

def bin_by_target_and_distractor(states, metadata, n_bins=12):
    """
    Bin trials by both target color and distractor color.
    
    Args:
        states: Neural states array [n_trials, 16]
        metadata: Metadata array [n_trials, 3] with [cue, color1, color2]
        n_bins: Number of angular bins per dimension
        
    Returns:
        Dictionary with binned states per cue
    """
    bin_size = 360.0 / n_bins
    
    # Initialize storage: cue -> (target_bin, distractor_bin) -> [states]
    binned_data = {
        1: {},
        2: {}
    }
    
    # Bin trials
    for i, (cue, c1, c2) in enumerate(metadata):
        cue = int(cue)
        
        # Determine target and distractor based on cue
        if cue == 1:
            target_angle = c1
            distractor_angle = c2
        else:
            target_angle = c2
            distractor_angle = c1
        
        target_bin = bin_angle(target_angle, bin_size)
        distractor_bin = bin_angle(distractor_angle, bin_size)
        
        key = (target_bin, distractor_bin)
        if key not in binned_data[cue]:
            binned_data[cue][key] = []
        binned_data[cue][key].append(states[i])
    
    # Average per bin
    averaged_data = {1: {}, 2: {}}
    bin_counts = {1: {}, 2: {}}
    
    for cue in [1, 2]:
        for key, states_list in binned_data[cue].items():
            if len(states_list) > 0:
                averaged_data[cue][key] = np.mean(states_list, axis=0)
                bin_counts[cue][key] = len(states_list)
    
    return averaged_data, bin_counts

# Bin the data
averaged_2d, bin_counts_2d = bin_by_target_and_distractor(states_prep2, metadata_prep2, N_BINS)

print(f"\nBinned by target × distractor (12 × 12 = 144 bins per cue)")
print(f"Cue 1: {len(averaged_2d[1])} non-empty bins")
print(f"Cue 2: {len(averaged_2d[2])} non-empty bins")

# Show some statistics
for cue in [1, 2]:
    counts = list(bin_counts_2d[cue].values())
    print(f"\nCue {cue} bin occupancy:")
    print(f"  Total bins with data: {len(counts)}")
    print(f"  Trials per bin - mean: {np.mean(counts):.1f}, min: {min(counts)}, max: {max(counts)}")

In [ ]:
# Visualisation of 2D binned data in PCA space
print("\n" + "="*60)
print("VISUALISATION: TARGET vs DISTRACTOR ENCODING (COMBINED)")
print("="*60)

# KEY INSIGHT: Combine both cues and create SEPARATE averages:
# 1. Average by TARGET only (collapsing across distractors AND cues) 
# 2. Average by DISTRACTOR only (collapsing across targets AND cues)
# Then run PCA on the combined set

# Use 12 bins (30° each) as in the original analysis
N_COLOR_BINS = 12  # 12 bins at 30° each

print("\nCreating separate averages for target and distractor (combining both cues)...")

# 1. Average by TARGET only (marginalizing over distractor and cue)
target_only_averaged = {}

for target_bin in range(N_BINS):
    # Collect all states with this target bin from BOTH cues
    states_for_target = []
    for cue in [1, 2]:
        for (fine_target, fine_distractor), state in averaged_2d[cue].items():
            if fine_target == target_bin:
                states_for_target.append(state)
    
    if len(states_for_target) > 0:
        target_only_averaged[target_bin] = np.mean(states_for_target, axis=0)

print(f"Target-only averaging (30° bins, both cues combined):")
print(f"  Total target bins: {len(target_only_averaged)}")

# 2. Average by DISTRACTOR only (marginalizing over target and cue)
distractor_only_averaged = {}

for distractor_bin in range(N_BINS):
    # Collect all states with this distractor bin from BOTH cues
    states_for_distractor = []
    for cue in [1, 2]:
        for (fine_target, fine_distractor), state in averaged_2d[cue].items():
            if fine_distractor == distractor_bin:
                states_for_distractor.append(state)
    
    if len(states_for_distractor) > 0:
        distractor_only_averaged[distractor_bin] = np.mean(states_for_distractor, axis=0)

print(f"Distractor-only averaging (30° bins, both cues combined):")
print(f"  Total distractor bins: {len(distractor_only_averaged)}")

# Combine target and distractor states for joint PCA
combined_states = []
combined_metadata = []  # (type, bin) where type: 0=target, 1=distractor

# Add target states
for target_bin, state in sorted(target_only_averaged.items()):
    combined_states.append(state)
    combined_metadata.append((0, target_bin))  # 0 = target

# Add distractor states
for distractor_bin, state in sorted(distractor_only_averaged.items()):
    combined_states.append(state)
    combined_metadata.append((1, distractor_bin))  # 1 = distractor

combined_states = np.array(combined_states)
combined_metadata = np.array(combined_metadata)

print(f"\nCombined states shape: {combined_states.shape}")
print(f"  {(combined_metadata[:, 0] == 0).sum()} target bins + {(combined_metadata[:, 0] == 1).sum()} distractor bins")

# Run PCA on combined set
pca_combined = PCA(n_components=3)
combined_pca = pca_combined.fit_transform(combined_states)

print(f"\nPCA on combined target+distractor:")
print(f"  Explained variance: {pca_combined.explained_variance_ratio_}")
print(f"  Cumulative: {pca_combined.explained_variance_ratio_.cumsum()}")

# Split back into target and distractor
target_mask = combined_metadata[:, 0] == 0
distractor_mask = combined_metadata[:, 0] == 1

target_pca_combined = combined_pca[target_mask]
target_bins_combined = combined_metadata[target_mask, 1].astype(int)

distractor_pca_combined = combined_pca[distractor_mask]
distractor_bins_combined = combined_metadata[distractor_mask, 1].astype(int)

print(f"\nFinal PCA projections:")
print(f"  Target: {target_pca_combined.shape}")
print(f"  Distractor: {distractor_pca_combined.shape}")

# Colour mapping (HSV for 12 bins)
def angle_to_colour(bin_idx, n_bins=12):
    """Convert angle bin (0-11) to HSV colour"""
    hue = bin_idx / n_bins
    return colorsys.hsv_to_rgb(hue, 0.9, 0.9)

# Create visualisation - 1 row × 3 columns format
fig = plt.figure(figsize=(18, 5))

target_pca_data = target_pca_combined
target_bins_data = target_bins_combined
distractor_pca_data = distractor_pca_combined
distractor_bins_data = distractor_bins_combined

# Plot 1: 3D view
ax = fig.add_subplot(131, projection='3d')

# Plot TARGET encoding (circles) with colors
for i in range(len(target_pca_data)):
    colour = angle_to_colour(target_bins_data[i], N_COLOR_BINS)
    ax.scatter(target_pca_data[i, 0], target_pca_data[i, 1], target_pca_data[i, 2],
              c=[colour], marker='o', s=150, alpha=0.8, 
              edgecolors='k', linewidth=2)

# Plot DISTRACTOR encoding (triangles) with colors
for i in range(len(distractor_pca_data)):
    colour = angle_to_colour(distractor_bins_data[i], N_COLOR_BINS)
    ax.scatter(distractor_pca_data[i, 0], distractor_pca_data[i, 1], distractor_pca_data[i, 2],
              c=[colour], marker='^', s=120, alpha=0.8,
              edgecolors='grey', linewidth=1.5)

ax.set_xlabel(f'PC1 ({pca_combined.explained_variance_ratio_[0]:.1%})', fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca_combined.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
ax.set_zlabel(f'PC3 ({pca_combined.explained_variance_ratio_[2]:.1%})', fontsize=12, fontweight='bold')
ax.set_title('3D View\n○ Target, △ Distractor', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: PC1-PC2 view
ax2 = fig.add_subplot(132)

# Plot TARGET and DISTRACTOR with colors
for i in range(len(target_pca_data)):
    colour = angle_to_colour(target_bins_data[i], N_COLOR_BINS)
    ax2.scatter(target_pca_data[i, 0], target_pca_data[i, 1],
               c=[colour], marker='o', s=150, alpha=0.8, edgecolors='k', linewidth=2)

for i in range(len(distractor_pca_data)):
    colour = angle_to_colour(distractor_bins_data[i], N_COLOR_BINS)
    ax2.scatter(distractor_pca_data[i, 0], distractor_pca_data[i, 1],
               c=[colour], marker='^', s=120, alpha=0.8, edgecolors='grey', linewidth=1.5)

ax2.set_xlabel(f'PC1 ({pca_combined.explained_variance_ratio_[0]:.1%})', fontsize=12, fontweight='bold')
ax2.set_ylabel(f'PC2 ({pca_combined.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
ax2.set_title('PC1 vs PC2', fontsize=14, fontweight='bold')
ax2.set_title('PC1 vs PC2', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# Plot 3: PC2-PC3 view  
ax3 = fig.add_subplot(133)

for i in range(len(target_pca_data)):
    colour = angle_to_colour(target_bins_data[i], N_COLOR_BINS)
    ax3.scatter(target_pca_data[i, 1], target_pca_data[i, 2],
               c=[colour], marker='o', s=150, alpha=0.8, edgecolors='k', linewidth=2)

for i in range(len(distractor_pca_data)):
    colour = angle_to_colour(distractor_bins_data[i], N_COLOR_BINS)
    ax3.scatter(distractor_pca_data[i, 1], distractor_pca_data[i, 2],
               c=[colour], marker='^', s=120, alpha=0.8, edgecolors='grey', linewidth=1.5)

ax3.set_xlabel(f'PC2 ({pca_combined.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
ax3.set_ylabel(f'PC3 ({pca_combined.explained_variance_ratio_[2]:.1%})', fontsize=12, fontweight='bold')
ax3.set_title('PC2 vs PC3', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

plt.tight_layout()
plt.savefig(RESULTS_DIR / '2d_binning_target_vs_distractor_combined.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualization saved to: {RESULTS_DIR / '2d_binning_target_vs_distractor_combined.png'}")
print(f"   Layout: 1 row × 3 columns (3D, PC1-PC2, PC2-PC3)")
print(f"   Using {N_COLOR_BINS} colour bins (30° each)")
print("   Large circles (○) = averaged by target colour (across all distractors and cues)")
print("   Small triangles (△) = averaged by distractor colour (across all targets and cues)")

In [ ]:
# Fit planes to target and distractor data
print("\n" + "="*60)
print("PLANE FITTING: TARGET vs DISTRACTOR")
print("="*60)

def fit_plane_to_data(data):
    """
    Fit a best 2D plane to 3D data points using PCA.
    
    Returns:
        normal_vector: Normal vector to the best-fitting plane
        center: Center point of the data
        planarity: Measure of how well data fits a plane (variance ratio)
        variance: Variance along each principal component
    """
    center = np.mean(data, axis=0)
    centered_data = data - center
    
    pca = PCA(n_components=3)
    pca.fit(centered_data)
    
    # Normal vector is the 3rd principal component (least variance)
    normal_vector = pca.components_[2]
    normal_vector = normal_vector / np.linalg.norm(normal_vector)
    
    # Planarity: ratio of variance in first 2 PCs to total
    variance = pca.explained_variance_ratio_
    planarity = variance[:2].sum()  # Variance in the plane
    
    return normal_vector, center, planarity, variance

def angle_between_planes_degrees(normal1, normal2):
    """Calculate angle between two planes in degrees."""
    cos_angle = np.abs(np.dot(normal1, normal2))
    cos_angle = min(1.0, max(0.0, cos_angle))  # Numerical stability
    angle = np.arccos(cos_angle) * 180 / np.pi
    return angle

# Fit planes to target and distractor data
print("\nFitting plane to target data...")
target_normal, target_center, target_planarity, target_variance = fit_plane_to_data(target_pca_combined)
print(f"  Normal: {target_normal}")
print(f"  Center: {target_center}")
print(f"  Planarity: {target_planarity:.2%}")
print(f"  Variance per PC: {target_variance}")

print("\nFitting plane to distractor data...")
distractor_normal, distractor_center, distractor_planarity, distractor_variance = fit_plane_to_data(distractor_pca_combined)
print(f"  Normal: {distractor_normal}")
print(f"  Center: {distractor_center}")  
print(f"  Planarity: {distractor_planarity:.2%}")
print(f"  Variance per PC: {distractor_variance}")

# Compute angle between planes
plane_angle = angle_between_planes_degrees(target_normal, distractor_normal)
print(f"\nAngle between planes: {plane_angle:.2f}°")

# Compute separation between centers
separation = np.linalg.norm(target_center - distractor_center)
print(f"Separation between centers: {separation:.4f}")

# Interpretation
if plane_angle < 10:
    interpretation = "Nearly parallel (co-planar)"
elif plane_angle > 80:
    interpretation = "Nearly orthogonal (independent)"
else:
    interpretation = "Oblique (partially shared)"
print(f"Interpretation: {interpretation}")


In [ ]:
# Plane Fitting Analysis Across All Prep Epochs (Colored Binning)
print("\n" + "="*70)
print("PLANE FITTING ANALYSIS: ACROSS ALL PREP EPOCHS")
print("="*70)

# Define prep indices to analyze
prep_indices = [0, 1, 2, 3]
prep_names = {
    0: "Cue Presentation",
    1: "Post-Cue / Delay 1 Start", 
    2: "Stimulus Presentation / Delay 1 End",
    3: "Post-Stimulus / Delay 2"
}

# Store results for all prep epochs
all_prep_results = {}

for prep_idx in prep_indices:
    print(f"\n{'='*70}")
    print(f"Processing prep_idx = {prep_idx}: {prep_names[prep_idx]}")
    print(f"{'='*70}")
    
    # Extract neural states for this prep_idx
    print(f"Extracting neural states...")
    neural_states_prep = []
    metadata_prep = []
    
    for i, trial in enumerate(trials):
        if i % 100 == 0 and i > 0:
            print(f"  Progress: {i}/{len(trials)}")
        try:
            state = extract_neural_state_from_model(trial, task, model, device, prep_idx=prep_idx)
            neural_states_prep.append(state)
            metadata_prep.append([trial['cue'], trial['color1_angle'], trial['color2_angle']])
        except Exception as e:
            print(f"  Trial {i} failed: {e}")
            break
    
    neural_states_prep = np.array(neural_states_prep)
    metadata_prep = np.array(metadata_prep)
    
    print(f"✓ Extracted {len(neural_states_prep)} states (shape: {neural_states_prep.shape})")
    
    # Bin by cued color
    averaged_states_prep, _ = bin_and_average_by_cued_color(neural_states_prep, metadata_prep, N_BINS)
    
    # Combine both cues
    all_binned_states_prep = np.vstack([averaged_states_prep[1], averaged_states_prep[2]])
    print(f"Combined binned states shape: {all_binned_states_prep.shape}")
    
    # Clean data (remove NaN bins)
    valid_mask_prep = ~np.isnan(all_binned_states_prep).any(axis=1)
    all_binned_states_clean_prep = all_binned_states_prep[valid_mask_prep]
    print(f"After cleaning: {all_binned_states_clean_prep.shape} (removed {(~valid_mask_prep).sum()} NaN bins)")
    
    # Perform PCA
    pca_prep = PCA(n_components=3)
    pca_coords_prep = pca_prep.fit_transform(all_binned_states_clean_prep)
    
    print(f"\nPCA explained variance: {pca_prep.explained_variance_ratio_}")
    print(f"Total variance captured: {pca_prep.explained_variance_ratio_.sum():.2%}")
    
    # Split back into cue 1 and cue 2 for plane fitting
    n_bins_cue1_prep = N_BINS  # Assuming all bins are present
    cue1_pca_prep = pca_coords_prep[:n_bins_cue1_prep]
    cue2_pca_prep = pca_coords_prep[n_bins_cue1_prep:n_bins_cue1_prep*2]
    
    # Fit planes to each cue's data
    cue1_normal_prep, cue1_center_prep, cue1_planarity_prep, cue1_variance_prep = fit_plane_to_data(cue1_pca_prep)
    cue2_normal_prep, cue2_center_prep, cue2_planarity_prep, cue2_variance_prep = fit_plane_to_data(cue2_pca_prep)
    
    # Compute angle between planes
    plane_angle_prep = angle_between_planes_degrees(cue1_normal_prep, cue2_normal_prep)
    
    # Compute separation between plane centers
    separation_prep = np.linalg.norm(cue1_center_prep - cue2_center_prep)
    
    # Store results
    all_prep_results[prep_idx] = {
        'cue1_normal': cue1_normal_prep,
        'cue2_normal': cue2_normal_prep,
        'cue1_center': cue1_center_prep,
        'cue2_center': cue2_center_prep,
        'cue1_planarity': cue1_planarity_prep,
        'cue2_planarity': cue2_planarity_prep,
        'plane_angle': plane_angle_prep,
        'separation': separation_prep,
        'pca': pca_prep,
        'cue1_pca': cue1_pca_prep,
        'cue2_pca': cue2_pca_prep,
        'explained_variance': pca_prep.explained_variance_ratio_
    }
    
    # Print summary
    print(f"\n{'─'*70}")
    print(f"RESULTS FOR PREP_IDX {prep_idx}:")
    print(f"{'─'*70}")
    print(f"Cue 1 plane:")
    print(f"  Normal: {cue1_normal_prep}")
    print(f"  Planarity: {cue1_planarity_prep:.2%}")
    print(f"  Variance: {cue1_variance_prep}")
    
    print(f"\nCue 2 plane:")
    print(f"  Normal: {cue2_normal_prep}")
    print(f"  Planarity: {cue2_planarity_prep:.2%}")
    print(f"  Variance: {cue2_variance_prep}")
    
    print(f"\nAngle between planes: {plane_angle_prep:.2f}°")
    print(f"Separation between centers: {separation_prep:.4f}")
    
    if plane_angle_prep < 10:
        interpretation = "Nearly parallel (co-planar)"
    elif plane_angle_prep > 80:
        interpretation = "Nearly orthogonal (independent)"
    else:
        interpretation = "Oblique (partially shared)"
    print(f"Interpretation: {interpretation}")

print(f"\n{'='*70}")
print("SUMMARY ACROSS ALL PREP EPOCHS")
print(f"{'='*70}")
print(f"\n{'Prep':<6} {'Name':<35} {'Angle':<10} {'Sep':<10} {'Cue1 Plan':<12} {'Cue2 Plan':<12}")
print(f"{'Idx':<6} {'':<35} {'(deg)':<10} {'':<10} {'(%)':<12} {'(%)':<12}")
print(f"{'-'*70}")

for prep_idx in prep_indices:
    res = all_prep_results[prep_idx]
    print(f"{prep_idx:<6} {prep_names[prep_idx]:<35} {res['plane_angle']:<10.2f} {res['separation']:<10.4f} "
          f"{res['cue1_planarity']*100:<12.1f} {res['cue2_planarity']*100:<12.1f}")

print(f"\n✓ Analysis complete for all {len(prep_indices)} prep epochs")

In [ ]:
# Create prospective memory visualization across all prep indices
# 4 rows x 3 columns: (3D, PC1 vs PC2, PC2 vs PC3) for each prep_idx
print("\n" + "="*70)
print("CREATING PROSPECTIVE MEMORY ALL PREP INDICES VISUALIZATION")
print("="*70)

from matplotlib.lines import Line2D

fig = plt.figure(figsize=(18, 20))

# Prep index names for titles
prep_title_names = {
    0: "prep_idx=0: End of CUE\n(Start of Delay 1)",
    1: "prep_idx=1: Before Stimulus\n(End of Delay 1)",
    2: "prep_idx=2: End of STIMULUS\n(Start of Delay 2)",
    3: "prep_idx=3: Before Response"
}

# Marker legend encodes cue only
cue_marker_handles = [
    Line2D([0], [0], marker='o', linestyle='None', color='w', markerfacecolor='lightgray', markeredgecolor='k', markersize=7, label='Cue 1'),
    Line2D([0], [0], marker='^', linestyle='None', color='w', markerfacecolor='lightgray', markeredgecolor='k', markersize=7, label='Cue 2'),
]

for row_idx, prep_idx in enumerate(prep_indices):
    res = all_prep_results[prep_idx]
    sep = res['separation']

    # Before stimulus onset (prep_idx 0,1), use neutral colors to avoid implying stimulus coding.
    # After stimulus onset (prep_idx 2,3), color by cued-color bin hue.
    if prep_idx < 2:
        cue1_colors = np.tile(np.array([[0.55, 0.55, 0.55]]), (res['cue1_pca'].shape[0], 1))
        cue2_colors = np.tile(np.array([[0.55, 0.55, 0.55]]), (res['cue2_pca'].shape[0], 1))
        color_label = 'neutral gray (pre-stimulus)'
    else:
        cue1_bins = np.arange(res['cue1_pca'].shape[0]) % N_BINS
        cue2_bins = np.arange(res['cue2_pca'].shape[0]) % N_BINS
        cue1_colors = np.array([angle_to_colour(int(b), N_BINS) for b in cue1_bins])
        cue2_colors = np.array([angle_to_colour(int(b), N_BINS) for b in cue2_bins])
        color_label = 'cued-color bin hue'

    # 3D subplot (column 1)
    ax_3d = fig.add_subplot(4, 3, row_idx * 3 + 1, projection='3d')

    ax_3d.scatter(
        res['cue1_pca'][:, 0], res['cue1_pca'][:, 1], res['cue1_pca'][:, 2],
        c=cue1_colors, marker='o', s=80, edgecolors='k', linewidths=1.0, alpha=0.9
    )
    ax_3d.scatter(
        res['cue2_pca'][:, 0], res['cue2_pca'][:, 1], res['cue2_pca'][:, 2],
        c=cue2_colors, marker='^', s=80, edgecolors='k', linewidths=1.0, alpha=0.9
    )

    var_ratio = res['explained_variance']
    ax_3d.set_xlabel(f'PC1 ({var_ratio[0]:.1%})', fontsize=10)
    ax_3d.set_ylabel(f'PC2 ({var_ratio[1]:.1%})', fontsize=10)
    ax_3d.set_zlabel(f'PC3 ({var_ratio[2]:.1%})', fontsize=10)
    ax_3d.set_title(f"{prep_title_names[prep_idx]}\nSeparation: {sep:.3f}", fontsize=11, fontweight='bold')
    ax_3d.legend(handles=cue_marker_handles, fontsize=9)
    ax_3d.grid(True, alpha=0.3)

    # PC1 vs PC2 subplot (column 2)
    ax_12 = fig.add_subplot(4, 3, row_idx * 3 + 2)
    ax_12.scatter(
        res['cue1_pca'][:, 0], res['cue1_pca'][:, 1],
        c=cue1_colors, marker='o', s=80, edgecolors='k', linewidths=1.0, alpha=0.9
    )
    ax_12.scatter(
        res['cue2_pca'][:, 0], res['cue2_pca'][:, 1],
        c=cue2_colors, marker='^', s=80, edgecolors='k', linewidths=1.0, alpha=0.9
    )
    ax_12.set_xlabel(f'PC1 ({var_ratio[0]:.1%})', fontsize=10, fontweight='bold')
    ax_12.set_ylabel(f'PC2 ({var_ratio[1]:.1%})', fontsize=10, fontweight='bold')
    ax_12.set_title(f'PC1 vs PC2 (color = {color_label})', fontsize=11, fontweight='bold')
    ax_12.legend(handles=cue_marker_handles, fontsize=9)
    ax_12.grid(True, alpha=0.3)
    ax_12.axis('equal')

    # PC2 vs PC3 subplot (column 3)
    ax_23 = fig.add_subplot(4, 3, row_idx * 3 + 3)
    ax_23.scatter(
        res['cue1_pca'][:, 1], res['cue1_pca'][:, 2],
        c=cue1_colors, marker='o', s=80, edgecolors='k', linewidths=1.0, alpha=0.9
    )
    ax_23.scatter(
        res['cue2_pca'][:, 1], res['cue2_pca'][:, 2],
        c=cue2_colors, marker='^', s=80, edgecolors='k', linewidths=1.0, alpha=0.9
    )
    ax_23.set_xlabel(f'PC2 ({var_ratio[1]:.1%})', fontsize=10, fontweight='bold')
    ax_23.set_ylabel(f'PC3 ({var_ratio[2]:.1%})', fontsize=10, fontweight='bold')
    ax_23.set_title(f'PC2 vs PC3 (color = {color_label})', fontsize=11, fontweight='bold')
    ax_23.legend(handles=cue_marker_handles, fontsize=9)
    ax_23.grid(True, alpha=0.3)
    ax_23.axis('equal')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'prospective_memory_all_prep_indices.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualization saved to: {RESULTS_DIR / 'prospective_memory_all_prep_indices.png'}")
print("   Layout: 4 rows × 3 columns")
print("   prep_idx 0-1: neutral gray (pre-stimulus)")
print("   prep_idx 2-3: color = cued-color bin hue")
print("   marker = cue")

In [ ]:
# Helper definitions required by full timeline statistics cell
def _average_role_bins_for_cue(states_t, metadata_arr, cue, role, n_bins=12):
    bin_size = 360.0 / n_bins
    cue_mask = metadata_arr[:, 0].astype(int) == cue
    states_cue = states_t[cue_mask]
    meta_cue = metadata_arr[cue_mask]

    bins = {b: [] for b in range(n_bins)}
    for st, (_, c1, c2) in zip(states_cue, meta_cue):
        if cue == 1:
            angle = c1 if role == 'target' else c2
        else:
            angle = c2 if role == 'target' else c1
        b = int(angle // bin_size) % n_bins
        bins[b].append(st)

    out = []
    for b in range(n_bins):
        if bins[b]:
            out.append(np.mean(bins[b], axis=0))

    if not out:
        return np.empty((0, states_t.shape[1]), dtype=np.float32)
    return np.asarray(out, dtype=np.float32)

def _pc12_variance_ratio(x):
    if x.shape[0] < 3:
        return np.nan
    n_components = min(3, x.shape[0], x.shape[1])
    pca_local = PCA(n_components=n_components)
    pca_local.fit(x)
    ratios = pca_local.explained_variance_ratio_
    return float(ratios[: min(2, len(ratios))].sum())

def _compute_timestep_plane_metrics(states_t, metadata_arr, cue, n_bins=12):
    target_avg = _average_role_bins_for_cue(states_t, metadata_arr, cue, role='target', n_bins=n_bins)
    distractor_avg = _average_role_bins_for_cue(states_t, metadata_arr, cue, role='distractor', n_bins=n_bins)

    if target_avg.shape[0] < 3 or distractor_avg.shape[0] < 3:
        return {
            'plane_angle_deg': np.nan,
            'planarity_cos': np.nan,
            'centroid_separation': np.nan,
            'target_var_pc12': np.nan,
            'distractor_var_pc12': np.nan,
            'combined_var_pc12': np.nan,
            'target_planarity': np.nan,
            'distractor_planarity': np.nan,
            'target_centroid_norm': np.nan,
            'distractor_centroid_norm': np.nan,
        }

    combined = np.vstack([target_avg, distractor_avg])
    pca_combined = PCA(n_components=3)
    coords = pca_combined.fit_transform(combined)

    t_n = target_avg.shape[0]
    target_pca = coords[:t_n]
    distractor_pca = coords[t_n:]

    target_normal, target_center, target_planarity, _ = fit_plane_to_data(target_pca)
    distractor_normal, distractor_center, distractor_planarity, _ = fit_plane_to_data(distractor_pca)

    theta_deg = angle_between_planes_degrees(target_normal, distractor_normal)
    planarity_cos = float(np.abs(np.cos(np.deg2rad(theta_deg))))
    centroid_sep = float(np.linalg.norm(target_center - distractor_center))

    return {
        'plane_angle_deg': float(theta_deg),
        'planarity_cos': planarity_cos,
        'centroid_separation': centroid_sep,
        'target_var_pc12': _pc12_variance_ratio(target_avg),
        'distractor_var_pc12': _pc12_variance_ratio(distractor_avg),
        'combined_var_pc12': float(pca_combined.explained_variance_ratio_[:2].sum()),
        'target_planarity': float(target_planarity),
        'distractor_planarity': float(distractor_planarity),
        'target_centroid_norm': float(np.linalg.norm(target_center)),
        'distractor_centroid_norm': float(np.linalg.norm(distractor_center)),
    }

print('Timestep metric helpers are ready.')

In [ ]:
# Full timeline plane statistics: prep (all epochs) + diffusion
print("\n" + "=" * 96)
print("FULL TIMELINE PLANE STATISTICS (PREP + DIFFUSION)")
print("=" * 96)

required_symbols = ['trials', 'task', 'model', 'device', '_compute_timestep_plane_metrics', 'N_BINS']
missing = [s for s in required_symbols if s not in globals()]
if missing:
    raise NameError(
        f"Missing required symbols: {missing}. Run earlier setup/forward cells first."
    )

def extract_full_timeline_from_model(trial, task, model, device, ablation_vector=None):
    with torch.no_grad():
        probe_features = torch.tensor([[trial['color1_angle'], trial['color2_angle']]]) * (np.pi / 180)
        report_features = torch.tensor([[trial['color1_angle'], trial['color2_angle']]]) * (np.pi / 180)

        override_stimulus_features = {
            'probe_features': probe_features,
            'report_features': report_features,
        }
        override_stimulus_cart_features = {}
        for k in override_stimulus_features.keys():
            override_stimulus_cart_features[f'{k}_cart'] = torch.stack(
                polar2cart(1.0, override_stimulus_features[k]), -1
            )

        override_stimulus_features_dict = {}
        for k, v in override_stimulus_features.items():
            override_stimulus_features_dict[k] = v
        for k, v in override_stimulus_cart_features.items():
            override_stimulus_features_dict[k] = v

        override_stimulus_features_dict['cued_item_idx'] = torch.tensor([trial['cue'] - 1])

        task_variable_dict = task.task_variable_gen.generate_variable_dict(
            batch_size=1,
            override_stimulus_features_dict=override_stimulus_features_dict,
        )

        trial_info = task.generate_trial_information(
            batch_size=1,
            num_samples=1,
            override_task_variable_information=task_variable_dict,
        )

        prep_network_inputs_device = []
        for inp in trial_info.prep_network_inputs:
            prep_network_inputs_device.append(inp.to(device) if isinstance(inp, torch.Tensor) else inp)

        diffusion_network_inputs_device = []
        for inp in trial_info.diffusion_network_inputs:
            diffusion_network_inputs_device.append(inp.to(device) if isinstance(inp, torch.Tensor) else inp)

        sample_kwargs = {
            'prep_network_inputs': prep_network_inputs_device,
            'diffusion_network_inputs': diffusion_network_inputs_device,
            'prep_epoch_durations': trial_info.prep_epoch_durations,
            'diffusion_epoch_durations': trial_info.diffusion_epoch_durations,
            'samples_shape': [1, 1],
            'noise_scaler': 1.0,
        }
        if ablation_vector is not None:
            sample_kwargs['ablation_vector'] = ablation_vector

        prep_dicts, samples_dict = model.generate_samples(**sample_kwargs)

        prep_segments = []
        prep_epoch_lengths = []
        for prep_idx, prep_dict in enumerate(prep_dicts):
            if 'preparatory_trajectory' not in prep_dict:
                raise KeyError(f"preparatory_trajectory missing in prep_dicts[{prep_idx}]")

            prep_traj = prep_dict['preparatory_trajectory'][0, 0, :, :16].detach().cpu().numpy().astype(np.float32)
            prep_segments.append(prep_traj)
            prep_epoch_lengths.append(int(prep_traj.shape[0]))

        prep_timeline = np.concatenate(prep_segments, axis=0)
        traj_key = 'embedded_sample_trajectory' if 'embedded_sample_trajectory' in samples_dict else 'sample_trajectory'
        diffusion_timeline = samples_dict[traj_key][0, 0, :, :16].detach().cpu().numpy().astype(np.float32)

        full_timeline = np.concatenate([prep_timeline, diffusion_timeline], axis=0)
        return full_timeline, prep_epoch_lengths, int(diffusion_timeline.shape[0])

if 'timeline_states_forward' not in globals() or 'timeline_meta_forward' not in globals():
    print('Extracting full prep+diffusion trajectories from forward pass...')
    traj_rows = []
    meta_rows = []
    prep_epoch_lengths_ref = None
    diffusion_steps_ref = None

    for i, trial in enumerate(trials):
        if i % 50 == 0:
            print(f'  Processing trial {i}/{len(trials)}')

        full_timeline_i, prep_epoch_lengths_i, diffusion_steps_i = extract_full_timeline_from_model(
            trial, task, model, device, ablation_vector=ACTIVE_ABLATION_VECTOR
        )

        if prep_epoch_lengths_ref is None:
            prep_epoch_lengths_ref = prep_epoch_lengths_i
            diffusion_steps_ref = diffusion_steps_i
        else:
            if prep_epoch_lengths_i != prep_epoch_lengths_ref:
                raise RuntimeError(
                    f'Prep epoch lengths vary across trials: {prep_epoch_lengths_ref} vs {prep_epoch_lengths_i}'
                )
            if diffusion_steps_i != diffusion_steps_ref:
                raise RuntimeError(
                    f'Diffusion steps vary across trials: {diffusion_steps_ref} vs {diffusion_steps_i}'
                )

        traj_rows.append(full_timeline_i)
        meta_rows.append([trial['cue'], trial['color1_angle'], trial['color2_angle']])

    timeline_states_forward = np.asarray(traj_rows, dtype=np.float32)
    timeline_meta_forward = np.asarray(meta_rows, dtype=np.float32)
    prep_epoch_lengths_forward = prep_epoch_lengths_ref
    diffusion_steps_forward = diffusion_steps_ref

print(f'timeline_states_forward shape: {timeline_states_forward.shape}')
print(f'timeline_meta_forward shape: {timeline_meta_forward.shape}')

PREP_TOTAL_STEPS = int(sum(prep_epoch_lengths_forward))
DIFFUSION_TOTAL_STEPS = int(diffusion_steps_forward)
TOTAL_STEPS = int(timeline_states_forward.shape[1])

if TOTAL_STEPS != PREP_TOTAL_STEPS + DIFFUSION_TOTAL_STEPS:
    raise RuntimeError(
        f'Total steps mismatch: total={TOTAL_STEPS}, prep={PREP_TOTAL_STEPS}, diffusion={DIFFUSION_TOTAL_STEPS}'
    )

print(f'Prep steps total: {PREP_TOTAL_STEPS}')
print(f'Diffusion steps total: {DIFFUSION_TOTAL_STEPS}')
print(f'Full timeline steps: {TOTAL_STEPS}')

phase_names = []
phase_bounds = [0]
cursor = 0
for i, dur in enumerate(prep_epoch_lengths_forward):
    phase_names.append(f'prep_idx={i}')
    cursor += int(dur)
    phase_bounds.append(cursor)
phase_names.append('diffusion')
phase_bounds.append(TOTAL_STEPS)

print('Timeline mapping:')
cursor = 0
for i, dur in enumerate(prep_epoch_lengths_forward):
    start = cursor
    end = cursor + int(dur) - 1
    print(f'  prep_idx={i}: steps {start}..{end} (duration={dur})')
    cursor += int(dur)
print(f'  diffusion: steps {cursor}..{TOTAL_STEPS - 1} (duration={DIFFUSION_TOTAL_STEPS})')

timestep_metrics_timeline = {1: [], 2: []}
for cue in [1, 2]:
    for t in range(TOTAL_STEPS):
        m = _compute_timestep_plane_metrics(
            states_t=timeline_states_forward[:, t, :],
            metadata_arr=timeline_meta_forward,
            cue=cue,
            n_bins=N_BINS,
        )
        timestep_metrics_timeline[cue].append(m)
print('Computed full timeline metrics for cue 1 and cue 2.')

x = np.arange(TOTAL_STEPS)
cue1_angle = np.asarray([m['plane_angle_deg'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue2_angle = np.asarray([m['plane_angle_deg'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue1_plan = np.asarray([m['planarity_cos'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue2_plan = np.asarray([m['planarity_cos'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue1_sep = np.asarray([m['centroid_separation'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue2_sep = np.asarray([m['centroid_separation'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue1_var = np.asarray([m['combined_var_pc12'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue2_var = np.asarray([m['combined_var_pc12'] for m in timestep_metrics_timeline[2]], dtype=np.float64)

cue1_var_t = np.asarray([m['target_var_pc12'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue1_var_d = np.asarray([m['distractor_var_pc12'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue1_plan_t = np.asarray([m['target_planarity'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue1_plan_d = np.asarray([m['distractor_planarity'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue1_norm_t = np.asarray([m['target_centroid_norm'] for m in timestep_metrics_timeline[1]], dtype=np.float64)
cue1_norm_d = np.asarray([m['distractor_centroid_norm'] for m in timestep_metrics_timeline[1]], dtype=np.float64)

cue2_var_t = np.asarray([m['target_var_pc12'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue2_var_d = np.asarray([m['distractor_var_pc12'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue2_plan_t = np.asarray([m['target_planarity'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue2_plan_d = np.asarray([m['distractor_planarity'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue2_norm_t = np.asarray([m['target_centroid_norm'] for m in timestep_metrics_timeline[2]], dtype=np.float64)
cue2_norm_d = np.asarray([m['distractor_centroid_norm'] for m in timestep_metrics_timeline[2]], dtype=np.float64)

phase_footer = ' | '.join([
    f'prep_idx={i}:{d}' for i, d in enumerate(prep_epoch_lengths_forward)
] + [f'diffusion:{DIFFUSION_TOTAL_STEPS}'])

def _add_phase_markings(ax):
    span_colors = ['#cde5ff', '#e8f3ff', '#dff7e2', '#f9f2d0', '#f5e4ff']
    for i, phase in enumerate(phase_names):
        x0 = phase_bounds[i] - 0.5
        x1 = phase_bounds[i + 1] - 0.5
        ax.axvspan(x0, x1, color=span_colors[i % len(span_colors)], alpha=0.18, zorder=0)

    for b in phase_bounds[1:-1]:
        ax.axvline(b - 0.5, color='k', linestyle='--', linewidth=0.9, alpha=0.45)

    centers = []
    for i in range(len(phase_names)):
        centers.append((phase_bounds[i] + phase_bounds[i + 1] - 1) / 2.0)

    secax = ax.secondary_xaxis('top')
    secax.set_xticks(centers)
    secax.set_xticklabels(phase_names, fontsize=8)
    secax.set_xlabel('Phase', fontsize=9)

    ax.set_xlim(-0.5, TOTAL_STEPS - 0.5)

def _add_footer(fig):
    fig.text(
        0.5, 0.01,
        f'Timeline steps 0..{TOTAL_STEPS - 1} | {phase_footer}',
        ha='center', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.78, edgecolor='gray')
    )

# Figure A: Cue1 vs Cue2 over full timeline
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
_add_footer(fig)

ax = axes[0, 0]
ax.plot(x, cue1_angle, color='tab:red', linewidth=2.0, label='Cue 1')
ax.plot(x, cue2_angle, color='tab:blue', linewidth=2.0, label='Cue 2')
ax.set_title('Plane Angle Over Full Timeline')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Angle (deg)')
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[0, 1]
ax.plot(x, cue1_plan, color='tab:red', linewidth=2.0, label='Cue 1')
ax.plot(x, cue2_plan, color='tab:blue', linewidth=2.0, label='Cue 2')
ax.set_title('Planarity Over Full Timeline (abs(cos(angle)))')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Planarity')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[1, 0]
ax.plot(x, cue1_sep, color='tab:red', linewidth=2.0, label='Cue 1')
ax.plot(x, cue2_sep, color='tab:blue', linewidth=2.0, label='Cue 2')
ax.set_title('Centroid Separation Over Full Timeline')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('L2 separation')
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[1, 1]
ax.plot(x, cue1_var, color='tab:red', linewidth=2.0, label='Cue 1')
ax.plot(x, cue2_var, color='tab:blue', linewidth=2.0, label='Cue 2')
ax.set_title('Combined PCA Variance (PC1+PC2)')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Variance explained')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

plt.tight_layout(rect=[0, 0.05, 1, 1])
out_path_timeline_cue_compare = RESULTS_DIR / 'timeline66_stats_cue1_vs_cue2_forward.png'
plt.savefig(out_path_timeline_cue_compare, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path_timeline_cue_compare}')

# Figure B: Cue1 target vs distractor over full timeline
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
_add_footer(fig)

ax = axes[0, 0]
ax.plot(x, cue1_angle, color='tab:green', linewidth=2.0)
ax.set_title('Cue 1: Plane Angle (Target vs Distractor)')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Angle (deg)')
ax.grid(alpha=0.3)
_add_phase_markings(ax)

ax = axes[0, 1]
ax.plot(x, cue1_plan_t, color='tab:green', linewidth=2.0, label='Target planarity')
ax.plot(x, cue1_plan_d, color='tab:orange', linewidth=2.0, label='Distractor planarity')
ax.set_title('Cue 1: Target vs Distractor Planarity')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Planarity')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[1, 0]
ax.plot(x, cue1_norm_t, color='tab:green', linewidth=2.0, label='Target centroid norm')
ax.plot(x, cue1_norm_d, color='tab:orange', linewidth=2.0, label='Distractor centroid norm')
ax.set_title('Cue 1: Target vs Distractor Centroid Norm')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Centroid norm')
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[1, 1]
ax.plot(x, cue1_var_t, color='tab:green', linewidth=2.0, label='Target variance (PC1+PC2)')
ax.plot(x, cue1_var_d, color='tab:orange', linewidth=2.0, label='Distractor variance (PC1+PC2)')
ax.set_title('Cue 1: Target vs Distractor PCA Variance')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Variance explained')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

plt.tight_layout(rect=[0, 0.05, 1, 1])
out_path_timeline_cue1 = RESULTS_DIR / 'timeline66_stats_cue1_target_vs_distractor_forward.png'
plt.savefig(out_path_timeline_cue1, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path_timeline_cue1}')

# Figure C: Cue2 target vs distractor over full timeline
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
_add_footer(fig)

ax = axes[0, 0]
ax.plot(x, cue2_angle, color='tab:purple', linewidth=2.0)
ax.set_title('Cue 2: Plane Angle (Target vs Distractor)')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Angle (deg)')
ax.grid(alpha=0.3)
_add_phase_markings(ax)

ax = axes[0, 1]
ax.plot(x, cue2_plan_t, color='tab:purple', linewidth=2.0, label='Target planarity')
ax.plot(x, cue2_plan_d, color='tab:brown', linewidth=2.0, label='Distractor planarity')
ax.set_title('Cue 2: Target vs Distractor Planarity')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Planarity')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[1, 0]
ax.plot(x, cue2_norm_t, color='tab:purple', linewidth=2.0, label='Target centroid norm')
ax.plot(x, cue2_norm_d, color='tab:brown', linewidth=2.0, label='Distractor centroid norm')
ax.set_title('Cue 2: Target vs Distractor Centroid Norm')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Centroid norm')
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

ax = axes[1, 1]
ax.plot(x, cue2_var_t, color='tab:purple', linewidth=2.0, label='Target variance (PC1+PC2)')
ax.plot(x, cue2_var_d, color='tab:brown', linewidth=2.0, label='Distractor variance (PC1+PC2)')
ax.set_title('Cue 2: Target vs Distractor PCA Variance')
ax.set_xlabel('Total step (prep + diffusion)')
ax.set_ylabel('Variance explained')
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
_add_phase_markings(ax)

plt.tight_layout(rect=[0, 0.05, 1, 1])
out_path_timeline_cue2 = RESULTS_DIR / 'timeline66_stats_cue2_target_vs_distractor_forward.png'
plt.savefig(out_path_timeline_cue2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path_timeline_cue2}')

In [ ]:
n_bins_global = N_BINS if "N_BINS" in globals() else 12


def get_states_and_metadata_for_prep(prep_idx):
    """Return (states, metadata) for a prep index (always fresh extraction, no caching)."""
    # Always extract fresh states to ensure ablation vectors are applied correctly
    states = []
    meta = []
    for i, trial in enumerate(trials):
        if i % 200 == 0 and i > 0:
            print(f"  prep_idx={prep_idx}: {i}/{len(trials)}")
        st = extract_neural_state_from_model(trial, task, model, device, prep_idx=prep_idx)
        states.append(st)
        meta.append([trial["cue"], trial["color1_angle"], trial["color2_angle"]])

    states = np.array(states)
    meta = np.array(meta)
    return states, meta

In [ ]:
# Student-only batch run for prospective-memory plots
import gc

_student_run_names = [
    "index_cued_first_diffusion_0.3_swap_recovery_ablation_0_0",
    "index_cued_first_diffusion_0.3_swap_recovery_ablation_1_0",
    "index_cued_first_diffusion_0.3_swap_recovery_ablation_4_0",
    "index_cued_first_diffusion_0.3_swap_recovery_ablation_5_0",
    "index_cued_first_diffusion_0.3_swap_recovery_ablation_6_0",
    "index_cued_first_diffusion_0.3_swap_ablation_7_1",
]

print(f"Student batch: {len(_student_run_names)} runs")

for _run_name in _student_run_names:
    _cfg = {
        "args_path": REPO_ROOT / "results_link_sampler" / _run_name / "args.yaml",
        "checkpoint_path": REPO_ROOT / "results_link_sampler" / _run_name / "state.mdl",
        "nullspace_json_path": None,
        "ablation_direction_idx": None,
    }

    print(f"\n{'='*80}\nSTUDENT BATCH: {_run_name}\n{'='*80}")

    _, _task_r, _model_r, _, _ = generate_model_and_task_from_args_path_multiepoch(
        str(_cfg["args_path"]), device
    )
    _ckpt = torch.load(_cfg["checkpoint_path"], map_location=device, weights_only=True)
    _model_r.load_state_dict(_ckpt)
    _model_r.eval()

    _results_dir = DATA_DIR / "results" / "prospective_memory_dual" / _run_name
    _results_dir.mkdir(parents=True, exist_ok=True)

    LOADED_MODELS[_run_name] = {
        "model": _model_r,
        "task": _task_r,
        "ablation_matrix": None,
        "results_dir": _results_dir,
    }

    extract_states_for_run(_run_name, prep_idx=PREP_IDX, force=True)
    run_full_analysis_for_active_run()

    LOADED_MODELS[_run_name]["model"] = None
    del _model_r, _ckpt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print(f"✓ Complete: {_run_name} -> {_results_dir}")

print("\n✓ Student-only batch complete.")

In [ ]:
# Run the healthy (unablated) teacher model
import gc

_teacher_run_name = "index_cued_first_diffusion_0.3_swap_7"
_teacher_path = REPO_ROOT / 'results_link_sampler' / _teacher_run_name

_, _task_r, _model_r, _, _ = generate_model_and_task_from_args_path_multiepoch(
    str(_teacher_path / 'args.yaml'), device
)
_ckpt = torch.load(_teacher_path / 'state.mdl', map_location=device, weights_only=True)
_model_r.load_state_dict(_ckpt)
_model_r.eval()

_results_dir = DATA_DIR / 'results' / 'prospective_memory_dual' / _teacher_run_name
_results_dir.mkdir(parents=True, exist_ok=True)

LOADED_MODELS[_teacher_run_name] = {
    'model': _model_r,
    'task': _task_r,
    'ablation_matrix': None,
    'results_dir': _results_dir,
}

extract_states_for_run(_teacher_run_name, prep_idx=PREP_IDX, force=True)
run_full_analysis_for_active_run()

LOADED_MODELS[_teacher_run_name]['model'] = None
del _model_r, _ckpt
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print(f"\u2713 Complete: {_teacher_run_name}  \u2192  {_results_dir}")


In [ ]:
# Auto-discover all ablation-recovery student runs and batch-process them
import gc, re

_pattern = re.compile(
    r"^index_cued_first_diffusion_0\.3_swap_recovery_ablation_(.+)_(\d+)$"
)

_sampler_dir = REPO_ROOT / "results_link_sampler"
_student_run_names = sorted(
    d.name
    for d in _sampler_dir.iterdir()
    if _pattern.match(d.name)
    and (d / "args.yaml").exists()
    and (d / "state.mdl").exists()
)

print(f"Found {len(_student_run_names)} student runs:")
for _n in _student_run_names:
    print(f"  {_n}")

for _run_name in _student_run_names:
    _cfg = {
        "args_path": _sampler_dir / _run_name / "args.yaml",
        "checkpoint_path": _sampler_dir / _run_name / "state.mdl",
    }

    _sep = "=" * 80
    print(f"\n{_sep}\nSTUDENT BATCH: {_run_name}\n{_sep}")

    _, _task_r, _model_r, _, _ = generate_model_and_task_from_args_path_multiepoch(
        str(_cfg["args_path"]), device
    )
    _ckpt = torch.load(_cfg["checkpoint_path"], map_location=device, weights_only=True)
    _model_r.load_state_dict(_ckpt)
    _model_r.eval()

    _results_dir = DATA_DIR / "results" / "prospective_memory_dual" / _run_name
    _results_dir.mkdir(parents=True, exist_ok=True)

    LOADED_MODELS[_run_name] = {
        "model": _model_r,
        "task": _task_r,
        "ablation_matrix": None,
        "results_dir": _results_dir,
    }

    extract_states_for_run(_run_name, prep_idx=PREP_IDX, force=True)
    activate_run(_run_name, restore_extracted=True)  # restore neural_states/metadata globals
    run_full_analysis_for_active_run()

    LOADED_MODELS[_run_name]["model"] = None
    del _model_r, _ckpt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print(f"\u2713 Complete: {_run_name}  \u2192  {_results_dir}")

print("\n\u2713 All student runs complete.")


In [ ]:
# Batch: run all ablated teachers that have at least one student NPZ.
# Generates timeline_raw_states.npz for each teacher so cell 24 can match them.
import gc, re as _re2

_FBASE2  = DATA_DIR / "results" / "prospective_memory_dual"
_PAT_STU = _re2.compile(
    r'^index_cued_first_diffusion_0\.3_swap_recovery_ablation_(.+)_(\d+)$'
)

# Discover which ablation dirs have at least one student NPZ
_needed_dirs = set()
for _d in _FBASE2.iterdir():
    _m = _PAT_STU.match(_d.name)
    if _m and (_d / 'timeline_raw_states.npz').exists():
        _key = _m.group(1)
        try: _key = int(_key)
        except ValueError: pass
        if _key != 'idk':
            _needed_dirs.add(_key)

print(f"Ablation dirs with students: {sorted(_needed_dirs, key=str)}")

for _dir_key in sorted(_needed_dirs, key=str):
    if _dir_key == 'no_ablation':
        _run_name   = 'index_cued_first_diffusion_0.3_swap_7'
        _abl_vec    = None
        print(f"\n{'='*80}\nTEACHER (no ablation): {_run_name}\n{'='*80}")
    else:
        _run_name = f"ablated_teacher_dir_{_dir_key:02d}"
        _abl_vec  = _load_single_nullspace_vector(NULLSPACE_JSON_PATH, _dir_key, device)
        print(f"\n{'='*80}\nTEACHER dir={_dir_key}: {_run_name}\n{'='*80}")

    _, _task_r, _model_r, _, _ = generate_model_and_task_from_args_path_multiepoch(
        str(TEACHER_ARGS_PATH), device
    )
    _ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device, weights_only=True)
    _model_r.load_state_dict(_ckpt)
    _model_r.eval()

    _results_dir = _FBASE2 / _run_name
    _results_dir.mkdir(parents=True, exist_ok=True)

    LOADED_MODELS[_run_name] = {
        'model':          _model_r,
        'task':           _task_r,
        'ablation_matrix': _abl_vec,
        'results_dir':    _results_dir,
    }

    extract_states_for_run(_run_name, prep_idx=PREP_IDX, force=True)
    run_full_analysis_for_active_run()

    LOADED_MODELS[_run_name]['model'] = None
    del _model_r, _ckpt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print(f"Done: {_run_name}  ->  {_results_dir}")

print("\nAll teacher runs complete. Now run cell 24.")


In [ ]:
# ── Separation fidelity: teacher vs recovery students (ring + timeline stats) ──
# Run AFTER all teacher/student runs have been processed (NPZs must exist).
import re as _re
from sklearn.decomposition import PCA as _FPCA

_FBASE   = DATA_DIR / "results" / "prospective_memory_dual"
_NB_F    = 12
_BINSZ_F = 30.0

# Metrics where signal = target_curve - distractor_curve
_TGTDIS_METRICS = ['mean_radius', 'radius_std', 'eccentricity', 'arc_std',
                   'planarity', 'centroid_norm', 'var_pc12']
# Metrics that are already a separation (single curve per cue)
_SEP_METRICS    = ['plane_angle_deg', 'centroid_separation', 'var_pc12_combined']
_ALL_METRICS    = _TGTDIS_METRICS + _SEP_METRICS


def separation_fidelity(teacher_tgt, teacher_dis, recovery_tgt, recovery_dis,
                        window=slice(20, None), center=False):
    s_t = (teacher_tgt - teacher_dis)[window]
    s_r = (recovery_tgt - recovery_dis)[window]
    if center:
        s_t = s_t - s_t.mean(); s_r = s_r - s_r.mean()
    return float(s_t @ s_r / (np.linalg.norm(s_t) * np.linalg.norm(s_r) + 1e-12))


def _group_bins(tl_mt):
    g = {c: {'target': {b: [] for b in range(_NB_F)},
              'distractor': {b: [] for b in range(_NB_F)}} for c in [1, 2]}
    for i, (cue, c1, c2) in enumerate(tl_mt):
        cue = int(cue)
        ta, da = (c1, c2) if cue == 1 else (c2, c1)
        g[cue]['target'][int(ta // _BINSZ_F) % _NB_F].append(i)
        g[cue]['distractor'][int(da // _BINSZ_F) % _NB_F].append(i)
    for c in [1, 2]:
        for role in ['target', 'distractor']:
            for b in range(_NB_F):
                g[c][role][b] = np.array(g[c][role][b], dtype=np.int64)
    return g


def _bin_avg(tl_st, idx_per_bin):
    T, D = tl_st.shape[1], tl_st.shape[2]
    out = np.zeros((_NB_F, T, D), np.float32)
    for b, idx in idx_per_bin.items():
        if len(idx):
            out[b] = tl_st[idx].mean(0)
    return out


def _ring_metrics_over_time(tgt_bm, dis_bm):
    # tgt_bm, dis_bm: (12, T, D)
    # Returns _TGTDIS_METRICS as (2, T) [tgt, dis] and _SEP_METRICS as (T,)
    T = tgt_bm.shape[1]
    mr   = np.full((2, T), np.nan, np.float32)
    rs   = np.full((2, T), np.nan, np.float32)
    ecc  = np.full((2, T), np.nan, np.float32)
    arc  = np.full((2, T), np.nan, np.float32)
    plan = np.full((2, T), np.nan, np.float32)
    cnrm = np.full((2, T), np.nan, np.float32)
    vpc  = np.full((2, T), np.nan, np.float32)
    pang = np.full(T, np.nan, np.float32)
    csep = np.full(T, np.nan, np.float32)
    vcom = np.full(T, np.nan, np.float32)

    for t in range(T):
        tp, dp = tgt_bm[:, t, :], dis_bm[:, t, :]
        comb   = np.vstack([tp, dp])
        cpca   = _FPCA(n_components=3)
        coords = cpca.fit_transform(comb)
        t3, d3 = coords[:_NB_F], coords[_NB_F:]
        vcom[t] = float(cpca.explained_variance_ratio_[:2].sum())

        tc, dc = t3.mean(0), d3.mean(0)
        td = np.linalg.norm(t3 - tc, axis=1)
        dd = np.linalg.norm(d3 - dc, axis=1)
        mr[0, t], mr[1, t]   = td.mean(), dd.mean()
        rs[0, t], rs[1, t]   = td.std(),  dd.std()
        cnrm[0, t]            = np.linalg.norm(tc)
        cnrm[1, t]            = np.linalg.norm(dc)
        csep[t]               = np.linalg.norm(tc - dc)

        nt = nd = None
        for ri, (pts, ctr) in enumerate([(t3, tc), (d3, dc)]):
            p = _FPCA(n_components=3).fit(pts - ctr)
            plan[ri, t] = float(p.explained_variance_ratio_[:2].sum())
            vpc[ri, t]  = float(_FPCA(n_components=2).fit(pts).explained_variance_ratio_.sum())
            if ri == 0: nt = p.components_[2]
            else:       nd = p.components_[2]

        pang[t] = float(np.degrees(np.arccos(np.clip(abs(np.dot(nt, nd)), 0, 1))))

        def _ecc(pts):
            ev = _FPCA(n_components=2).fit(pts).explained_variance_
            return ev[0] / ev[1] if ev[1] > 1e-12 else np.nan
        ecc[0, t], ecc[1, t] = _ecc(t3), _ecc(d3)

        def _arc(pts):
            return float(np.std([np.linalg.norm(pts[(i+1)%_NB_F]-pts[i]) for i in range(_NB_F)]))
        arc[0, t], arc[1, t] = _arc(t3), _arc(d3)

    return {'mean_radius': mr, 'radius_std': rs, 'eccentricity': ecc, 'arc_std': arc,
            'planarity': plan, 'centroid_norm': cnrm, 'var_pc12': vpc,
            'plane_angle_deg': pang, 'centroid_separation': csep, 'var_pc12_combined': vcom}


def _metric_fidelity(tm, sm, m, window, center):
    if m in _TGTDIS_METRICS:
        return separation_fidelity(tm[m][0], tm[m][1], sm[m][0], sm[m][1], window, center)
    z = np.zeros(tm[m].shape, np.float32)
    return separation_fidelity(tm[m], z, sm[m], z, window, center)


def _full_metrics_both_cues(npz):
    g = _group_bins(npz['tl_mt'])
    return g, {c: _ring_metrics_over_time(_bin_avg(npz['tl_st'], g[c]['target']),
                                           _bin_avg(npz['tl_st'], g[c]['distractor']))
               for c in [1, 2]}


def _resample_bins(g, c, rng):
    out = {}
    for role in ['target', 'distractor']:
        out[role] = {}
        for b in range(_NB_F):
            idx = g[c][role][b]
            out[role][b] = (rng.choice(idx, size=len(idx), replace=True)
                            if len(idx) else np.array([], dtype=np.int64))
    return out


def _half_split(g, c, rng):
    h1, h2 = {}, {}
    for role in ['target', 'distractor']:
        h1[role], h2[role] = {}, {}
        for b in range(_NB_F):
            idx = g[c][role][b]
            if len(idx) < 2:
                h1[role][b] = h2[role][b] = idx
            else:
                perm = rng.permutation(idx)
                mid  = len(perm) // 2
                h1[role][b], h2[role][b] = perm[:mid], perm[mid:]
    return h1, h2


def compute_fidelity_pair(tch_npz_path, stu_npz_path,
                           window=slice(20, None), center=False,
                           n_boot=0, seed=42):
    tch = np.load(tch_npz_path)
    stu = np.load(stu_npz_path)
    rng = np.random.default_rng(seed)

    tch_g, tch_m = _full_metrics_both_cues(tch)
    stu_g, stu_m = _full_metrics_both_cues(stu)

    def _avg_cues(tm, sm, m):
        return float(np.mean([_metric_fidelity(tm[c], sm[c], m, window, center)
                               for c in [1, 2]]))

    point = {m: _avg_cues(tch_m, stu_m, m) for m in _ALL_METRICS}

    if n_boot == 0:
        return {m: {'fidelity': point[m]} for m in _ALL_METRICS}

    boot  = {m: [] for m in _ALL_METRICS}
    noise = {m: [] for m in _ALL_METRICS}

    for _ in range(n_boot):
        bv, nv = {m: [] for m in _ALL_METRICS}, {m: [] for m in _ALL_METRICS}
        for c in [1, 2]:
            bt = _resample_bins(tch_g, c, rng)
            bs = _resample_bins(stu_g, c, rng)
            bt_m = _ring_metrics_over_time(_bin_avg(tch['tl_st'], bt['target']),
                                            _bin_avg(tch['tl_st'], bt['distractor']))
            bs_m = _ring_metrics_over_time(_bin_avg(stu['tl_st'], bs['target']),
                                            _bin_avg(stu['tl_st'], bs['distractor']))
            h1, h2 = _half_split(tch_g, c, rng)
            h1_m = _ring_metrics_over_time(_bin_avg(tch['tl_st'], h1['target']),
                                            _bin_avg(tch['tl_st'], h1['distractor']))
            h2_m = _ring_metrics_over_time(_bin_avg(tch['tl_st'], h2['target']),
                                            _bin_avg(tch['tl_st'], h2['distractor']))
            for m in _ALL_METRICS:
                bv[m].append(_metric_fidelity(bt_m, bs_m, m, window, center))
                nv[m].append(_metric_fidelity(h1_m, h2_m, m, window, center))
        for m in _ALL_METRICS:
            boot[m].append(float(np.mean(bv[m])))
            noise[m].append(float(np.mean(nv[m])))

    return {m: {'fidelity':    point[m],
                'ci_low':      float(np.percentile(boot[m], 2.5)),
                'ci_high':     float(np.percentile(boot[m], 97.5)),
                'noise_floor': float(np.mean(noise[m]))}
            for m in _ALL_METRICS}


# ── name parsing and matching ─────────────────────────────────────────────────

def _parse_run(name):
    m = _re.match(r'^ablated_teacher_dir_(\d+)$', name)
    if m: return ('teacher', int(m.group(1)))
    if name == 'index_cued_first_diffusion_0.3_swap_7':
        return ('teacher', 'no_ablation')
    m = _re.match(r'^index_cued_first_diffusion_0\.3_swap_recovery_ablation_(.+)_(\d+)$', name)
    if m:
        key = m.group(1)
        try: key = int(key)
        except ValueError: pass
        if key == 'idk': return ('unknown',)
        return ('student', key, int(m.group(2)))
    return ('unknown',)


def _build_match_table(base):
    # Teachers: prefer specific ablated-teacher NPZ; fall back to healthy teacher
    # for all students if no specific teacher NPZ is available.
    teachers, students = {}, {}
    healthy_teacher_npz = None
    for d in base.iterdir():
        npz = d / 'timeline_raw_states.npz'
        if not npz.exists(): continue
        t = _parse_run(d.name)
        if t[0] == 'teacher':
            teachers[t[1]] = npz
            if t[1] == 'no_ablation':
                healthy_teacher_npz = npz
        elif t[0] == 'student':
            students.setdefault(t[1], []).append((d.name, npz))

    if healthy_teacher_npz is None and not teachers:
        print("[error] No teacher NPZ found. Re-run cell 22 (healthy teacher) first.")
        return {}

    matched = {}
    for dk, stu_list in students.items():
        if dk in teachers:
            tch_npz = teachers[dk]
        elif healthy_teacher_npz is not None:
            tch_npz = healthy_teacher_npz
            if dk != 'no_ablation':
                print(f"  [info] dir={dk}: no specific teacher NPZ, using healthy teacher")
        else:
            print(f"  [warn] dir={dk}: no teacher NPZ at all, skipping")
            continue
        matched[dk] = {'teacher_npz': tch_npz, 'students': sorted(stu_list)}
    return matched


# ── run ───────────────────────────────────────────────────────────────────────
_WINDOW = slice(20, None)
_N_BOOT = 200

print("Scanning for saved NPZ files ...")
_match = _build_match_table(_FBASE)
print(f"Found {len(_match)} matched teacher-student group(s):\n"
      + '\n'.join(f"  dir={k}: {len(v['students'])} student(s)" for k, v in _match.items()))

_all_results = {}
for _dk, _grp in sorted(_match.items(), key=lambda x: str(x[0])):
    for _sname, _snpz in _grp['students']:
        print(f"\ndir={_dk}  |  {_sname}")
        _res = compute_fidelity_pair(_grp['teacher_npz'], _snpz, window=_WINDOW, n_boot=_N_BOOT)
        _all_results.setdefault(_dk, {})[_sname] = _res
        for _m, _s in _res.items():
            _ci_str = f"  [{_s.get('ci_low', np.nan):+.3f}, {_s.get('ci_high', np.nan):+.3f}]" if 'ci_low' in _s else ""
            _nf_str = f"  floor={_s.get('noise_floor', np.nan):.3f}" if 'noise_floor' in _s else ""
            print(f"  {_m:22s}  {_s['fidelity']:+.3f}{_ci_str}{_nf_str}")

# ── summary plots (10 separate figures, one per metric) ─────────────────────────
_pairs   = [(dk, sn, _all_results[dk][sn])
            for dk in sorted(_all_results, key=str)
            for sn in sorted(_all_results[dk])]

if _pairs:
    _x       = np.arange(len(_pairs))
    _xlabels = [f"dir={dk}\n{sn.split('ablation_')[-1]}" for dk, sn, _ in _pairs]

    for _mi, m in enumerate(_ALL_METRICS):
        fig, ax = plt.subplots(figsize=(14, 5))
        fids  = np.array([r[m]['fidelity']    for _, _, r in _pairs])
        _has_ci = 'ci_low' in _pairs[0][2][m] if _pairs else False
        
        ax.bar(_x, fids, color='steelblue', alpha=0.75, zorder=3, label='fidelity')
        if _has_ci:
            ci_lo = np.array([r[m]['ci_low']      for _, _, r in _pairs])
            ci_hi = np.array([r[m]['ci_high']     for _, _, r in _pairs])
            nf    = np.array([r[m]['noise_floor'] for _, _, r in _pairs])
            yerr  = np.array([fids - ci_lo, ci_hi - fids])
            ax.errorbar(_x, fids, yerr=yerr, fmt='none', color='k', capsize=4, lw=1.5, zorder=4)
            ax.scatter(_x, nf, color='tab:orange', marker='_', s=200, lw=2, zorder=5, label='noise floor')
        ax.axhline(0, color='gray', lw=0.7, linestyle=':')
        ax.axhline(1, color='gray', lw=0.7, linestyle=':')
        _kind = '(sep)' if m in _SEP_METRICS else '(tgt-dis)'
        ax.set_title(f"{m.replace('_', ' ')} {_kind}", fontsize=12, fontweight='bold')
        ax.set_xticks(_x)
        ax.set_xticklabels(_xlabels, rotation=45, ha='right', fontsize=9)
        ax.set_ylim(-0.15, 1.1)
        ax.set_ylabel('Cosine similarity', fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3, zorder=0)
        plt.tight_layout()
        
        _fig_path = _FBASE / f'fidelity_metric_{_mi:02d}_{m}.png'
        plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {_fig_path}")
    
    print(f"\n✓ Saved 10 metric plots to {_FBASE}/")


